<a href="https://colab.research.google.com/github/Heba2627/NLP-Project/blob/main/NLP_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 36

## Data

In [1]:
import pickle
from pathlib import Path
# -----------------------------------------------------------------------------------------------------------------------
# Checkpointing utilities:
# A handful of cells below are expensive (downloading data, running a translation model over thousands of questions,
# training the QC classifier and the RNN, grid-searching the n-gram models) but their input never changes between runs.
# Each such cell now checks whether its result is already on disk and loads it instead of recomputing; the first full run
# pays the full cost once and writes these files, every run after that is fast.
# ------------------------------------------------------------------------------------------------------------------------

CACHE_DIR = Path("checkpoints")
CACHE_DIR.mkdir(exist_ok=True)

def checkpoint_path(name):
    return CACHE_DIR / f"{name}.pkl"

def checkpoint_exists(name):
    return checkpoint_path(name).exists()

def save_checkpoint(name, obj):
    with open(checkpoint_path(name), "wb") as f:
        pickle.dump(obj, f)

def load_checkpoint(name):
    with open(checkpoint_path(name), "rb") as f:
        return pickle.load(f)

def clear_checkpoint(name):
    checkpoint_path(name).unlink(missing_ok=True)

In [2]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

if checkpoint_exists("raw_df"):
    df = load_checkpoint("raw_df")
else:
    splits = {'train': 'train.parquet', 'validation': 'validation.parquet', 'test': 'test.parquet'}
    dfs=[]
    for splitname,path in splits.items():
        df=pd.read_parquet("hf://datasets/coastalcph/tydi_xor_rc/"+ path)
        df["split"]=splitname
        dfs.append(df)

    df=pd.concat(dfs, ignore_index=True)
    save_checkpoint("raw_df", df)

In [3]:
df=df[df["lang"].isin(["ar", "ko", "te"])]
print(df.columns)

Index(['question', 'context', 'lang', 'answerable', 'answer_start', 'answer',
       'answer_inlang', 'split'],
      dtype='object')


In [4]:
answerable_df = df[df["answerable"] == True]
unanswerable_df = df[df["answerable"] == False]

print(f"Number of answerable instances: {len(answerable_df)}")
print(f"Number of unanswerable instances: {len(unanswerable_df)}")
print(f"{len(answerable_df)/len(df)*100:.2f}% of dataset that is answerable")
print(f"{len(unanswerable_df)/len(df)*100:.2f}% of dataset that is unanswerable")

Number of answerable instances: 6963
Number of unanswerable instances: 527
92.96% of dataset that is answerable
7.04% of dataset that is unanswerable


In [5]:
train_df= df[df["split"] == "train"]
val_df= df[df["split"] == "validation"]
print(f"Train split size: {len(train_df)}")
print(f"Validation split size: {len(val_df)}")

Train split size: 6335
Validation split size: 1155


In [6]:
import regex as re
word_pattern = re.compile(r"[\p{L}\p{M}]+", flags=re.UNICODE)

def n_tokens(text):
    return len(word_pattern.findall(str(text)))

df["question_len"] = df["question"].apply(n_tokens)
df["context_len"] = df["context"].apply(n_tokens)


def median_iqr(s):
    q1, median, q3 = np.percentile(s, [25, 50, 75])
    return median, q3-q1

def duplicate_counts(data):
    duplicated =data.duplicated(subset=["question", "context"], keep="first")
    duplicated_all =data.duplicated(subset=["question", "context"], keep=False)
    return duplicated.sum(), duplicated_all.sum()

def summarize_group(data, split_name, lang_name):
    q_med,q_iqr= median_iqr(data["question_len"])
    c_med,c_iqr= median_iqr(data["context_len"])

    dup_redundant,dup_in_groups= duplicate_counts(data)

    answerable= data["answerable"].eq(True).sum()
    unanswerable= data["answerable"].eq(False).sum()
    n= len(data)

    return {
        "Split": split_name,
        "Language": lang_name,
        "N": n,
        "Answerable": answerable,
        "Unanswerable": unanswerable,
        "% Answerable": 100 * answerable / n if n else np.nan,
        "% Unanswerable": 100 * unanswerable / n if n else np.nan,
        "Question median (IQR)": (f"{q_med:.1f} ({q_iqr:.1f})" if pd.notna(q_med) else "NA"),
        "Context median (IQR)": (f"{c_med:.1f} ({c_iqr:.1f})" if pd.notna(c_med) else "NA"),
        "Missing answer_inlang": (data["answer_inlang"].isna().sum() if "answer_inlang" in data.columns else np.nan),
        "Rows with any missing": int(data.isna().any(axis=1).sum()),
        "Duplicate rows (redundant)": int(dup_redundant),
        "Rows in duplicate groups": int(dup_in_groups),
    }
rows= []
for split_name, split_data in df.groupby("split", sort=False):
    for lang_name, lang_data in split_data.groupby("lang", sort=False):
        rows.append(summarize_group(lang_data, split_name, lang_name))

    # All languages within each split
    rows.append(summarize_group(split_data, split_name, "All languages"))

# Entire dataset (all languages and all splits)
rows.append(summarize_group(df, "Overall", "All languages"))
summary = pd.DataFrame(rows)

print("CORPUS STATISTICS")
print(summary.to_string(index=False))

CORPUS STATISTICS
     Split      Language    N  Answerable  Unanswerable  % Answerable  % Unanswerable Question median (IQR) Context median (IQR)  Missing answer_inlang  Rows with any missing  Duplicate rows (redundant)  Rows in duplicate groups
     train            ko 2422        2359            63     97.398844        2.601156             5.0 (2.0)          82.0 (63.0)                   2422                   2422                          10                        20
     train            ar 2558        2303           255     90.031274        9.968726             6.0 (2.0)          86.5 (67.0)                   2558                   2558                           0                         0
     train            te 1355        1310            45     96.678967        3.321033             5.0 (2.0)          77.0 (60.0)                   1305                   1305                           0                         0
     train All languages 6335        5972           363     94.269

In [8]:
from transformers import AutoTokenizer,AutoModelForSeq2SeqLM
warnings.filterwarnings('ignore')

model_name = "facebook/nllb-200-distilled-600M"
tok = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to("cuda")

LANG_MAP = {"ar": "arb_Arab","ko": "kor_Hang","te": "tel_Telu"}
def translate(token, lang):
    try:
        tok.src_lang = LANG_MAP[lang]
        inputs = tok(token, return_tensors="pt").to("cuda")
        output = model.generate(**inputs,forced_bos_token_id=tok.convert_tokens_to_ids("eng_Latn"))
        return tok.decode(output[0], skip_special_tokens=True)

    except Exception as e:
        print(f"failed: {token!r} ({lang}) -> {type(e).__name__}: {e}")
        return None

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

In [9]:
if checkpoint_exists("translation_cache"):
    translation_cache = load_checkpoint("translation_cache")
    print(f"Loaded {len(translation_cache)} cached translations.")
else:
    translation_cache = {}

def translate_cached(text, lang):
    key = (text, lang)
    if key in translation_cache:
        return translation_cache[key]
    result = translate(text, lang)
    translation_cache[key] = result
    return result

In [10]:
unique_questions = df[["question", "lang"]].drop_duplicates()
for q, lang in zip(unique_questions["question"], unique_questions["lang"]):
    translate_cached(q, lang)

# translate_cached already skips the model call for anything seen on a
# previous run; this just persists whatever new entries were added this run.
save_checkpoint("translation_cache", translation_cache)

In [11]:
from collections import Counter, defaultdict

#We need to root out symbols so check for letter
has_letter = re.compile(r"\p{L}", flags=re.UNICODE)
def tokenize_words_xlmr(text):
    pieces = tok.tokenize(str(text))
    words, current = [], ""
    for p in pieces:
        if p.startswith("▁"):
            if current != "":
                words.append(current)
            current = p[1:]
        else:
            current += p
    if current != "":
        words.append(current)
    return [w.lower() for w in words if w != "" and has_letter.search(w)]

counters = defaultdict(Counter)
for lang, question in zip(df["lang"], df["question"]):
    counters[lang].update(tokenize_words_xlmr(question))

rows = [ {"lang": lang, "token": token, "count": count}
    for lang, counter in counters.items()
    for token, count in counter.most_common(5)]
top5_df = pd.DataFrame(rows)
top5_df = top5_df[top5_df["lang"].isin(["ar", "ko", "te"])]
top5_df["In English"] = [translate(token, lang) for token, lang in zip(top5_df["token"], top5_df["lang"])]
top5_df = top5_df[["lang", "token", "In English", "count"]]
top5_df

,lang,token,In English,count
0,ko,가장,The most,593
1,ko,무엇인가?,What?,570
2,ko,언제,When?,380
3,ko,어디인가?,Where are you?,257
4,ko,몇,A few.,255
5,ar,من,Who ?,697
6,ar,في,In the,682
7,ar,متى,When ?,600
8,ar,ما,What ?,522
9,ar,هو,It is .,415


In [12]:
#Task 3
def verify_answerable(df):
    checked = 0
    failures = []
    for i, row in enumerate(df):
        if row["answerable"]:
            checked+= 1
            context=row["context"]
            answer=row["answer"]
            start=row["answer_start"]

            expected=context[start:start+len(answer)]
            if expected!=answer:
                failures.append({
                    "index": i,
                    "answer_start": start,
                    "answer": answer,
                    "substring_found": expected,
                    "context": context})
    return checked,failures

checked_train,failures_train=verify_answerable(train_df.to_dict("records"))
print(f"Checked {checked_train} answerable rows in the training set, found {len(failures_train)} failures")

checked_val,failures_val=verify_answerable(val_df.to_dict("records"))
print(f"Checked {checked_val} answerable rows in the validation set, found {len(failures_val)} failures")


Checked 5972 answerable rows in the training set, found 0 failures
Checked 991 answerable rows in the validation set, found 0 failures


## Baselines

### i) Majority Baseline

In [13]:
majority_label = df["answerable"].value_counts().idxmax()
print("Majority class form training split: ", majority_label)

def majority_baseline(question=None, Context=None):
    return majority_label

Majority class form training split:  True


In [14]:
majority_val_pred = [majority_label for _ in range(len(val_df))]
acc_score = (pd.Series(majority_val_pred).values == val_df["answerable"].values).mean()
print("Accuracy of Majority Baseline on Validation split: ", acc_score*100, "%")
#Note -> Final output & Analysis will be found with week 39 models comparisons

Accuracy of Majority Baseline on Validation split:  85.8008658008658 %


### ii) Determinstic rule-based classifier baseline

In [15]:
import string
from sklearn.metrics import accuracy_score, precision_score, recall_score,f1_score, confusion_matrix
from transformers.utils import logging
logging.set_verbosity_error()
warnings.filterwarnings("ignore")

tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-base")
STOPWORDS = {
    "the", "a", "an", "is", "are", "was", "were", "of", "in", "on", "at",
    "to", "for", "and", "or", "that", "this", "these", "those", "by", "with",
    "from", "as", "it", "its"}
QUESTION_TYPE_PATTERNS = {
    "person": re.compile(r"\bwho\b", re.I),
    "date": re.compile(r"\bwhen\b", re.I),
    "location": re.compile(r"\bwhere\b", re.I),
    "number": re.compile(r"\bhow many\b|\bhow much\b", re.I),}
ENTITY_PATTERNS = {
    "person": re.compile(r"\b[A-Z][a-z]+(?:\s[A-Z][a-z]+)+\b"),
    "date": re.compile(r"\b\d{1,2}(st|nd|rd|th)?\s\w+\s\d{4}\b|\b\d{4}\b|\b\w+\s\d{1,2},\s\d{4}\b"),
    "location": re.compile(r"\b[A-Z][a-z]+\b"),
    "number": re.compile(r"\b\d+([.,]\d+)?\b"),}


def clean_tokens(text):
    tokens= [t.lstrip("\u2581") for t in tokenizer.tokenize(text.lower())]
    return {t for t in tokens if t and t not in STOPWORDS and not all(c in string.punctuation for c in t)}



def lexical_overlap_score(question_en, context):
    q_tokens = clean_tokens(question_en)
    c_tokens = clean_tokens(context)
    if not q_tokens:
        return 0.0
    return len(q_tokens & c_tokens) / len(q_tokens)


def detect_question_type(question_en):
    for qtype, pattern in QUESTION_TYPE_PATTERNS.items():
        if pattern.search(question_en):
            return qtype
    return "other"


def context_has_type_match(context, qtype):
    if qtype == "other":
        return False
    pattern = ENTITY_PATTERNS.get(qtype)
    return bool(pattern.search(context)) if pattern else False


def extract_features(df):
    rows = []
    for r in df.to_dict("records"):
        q_en = translate_cached(r["question"], lang=r["lang"])
        qtype = detect_question_type(q_en)
        rows.append({
            "overlap": lexical_overlap_score(q_en, r["context"]),
            "has_type": qtype != "other",
            "type_match": context_has_type_match(r["context"], qtype),
        })
    return pd.DataFrame(rows, index=df.index)


def metrics_for(y_true, y_pred):
    return {
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "f1_answerable": f1_score(y_true, y_pred, pos_label=True, zero_division=0),
        "f1_unanswerable": f1_score(y_true, y_pred, pos_label=False, zero_division=0),
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "confusion_matrix": confusion_matrix(y_true, y_pred, labels=[False, True]).tolist(),
    }


def predict_from_features(feats, threshold, mode="or"):
    ov= feats["overlap"].to_numpy() >= threshold
    tm= feats["type_match"].to_numpy()
    ht= feats["has_type"].to_numpy()
    if mode == "overlap":
        return ov
    if mode == "or":
        return ov | tm
    if mode == "and_typed":
        return np.where(ht, ov & tm, ov)
    raise ValueError(mode)


val_feats= extract_features(val_df)
y_val= val_df["answerable"].astype(bool).to_numpy()
candidates= np.unique(np.concatenate([val_feats["overlap"].to_numpy(), [1.01]]))

sweep = []
for mode in ["overlap", "or", "and_typed"]:
    for t in candidates:
        pred = predict_from_features(val_feats, t, mode)
        m = metrics_for(y_val, pred)
        sweep.append({"mode": mode, "threshold": t, "pred_rate": pred.mean(),
                      "macro_f1": m["macro_f1"], "f1_ans": m["f1_answerable"],
                      "f1_unans": m["f1_unanswerable"], "acc": m["accuracy"]})
sweep_df= pd.DataFrame(sweep)

#Pick best F1 and ties go to the lowest threshold.
best= sweep_df.sort_values(["macro_f1", "threshold"], ascending=[False, True]).iloc[0]
BEST_MODE, BEST_THRESHOLD= best["mode"], float(best["threshold"])
print(f"Best on validation: mode={BEST_MODE}, threshold={BEST_THRESHOLD:.3f}, "
      f"macro-F1={best['macro_f1']:.3f}")
print("Best threshold per mode:")
print(sweep_df.sort_values("macro_f1", ascending=False)
      .groupby("mode").head(1).round(3).to_string(index=False))

final_pred = predict_from_features(val_feats, BEST_THRESHOLD, BEST_MODE)
print(f"Predicted-answerable rate: {final_pred.mean():.3f} | true rate: {y_val.mean():.3f}")
print(f"Final validation metrics: {metrics_for(y_val, final_pred)}")

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

Best on validation: mode=or, threshold=0.312, macro-F1=0.599
Best threshold per mode:
     mode  threshold  pred_rate  macro_f1  f1_ans  f1_unans   acc
       or      0.333      0.862     0.599   0.888     0.310 0.807
  overlap      0.286      0.807     0.567   0.855     0.279 0.758
and_typed      0.286      0.774     0.552   0.835     0.268 0.731
Predicted-answerable rate: 0.862 | true rate: 0.858
Final validation metrics: {'macro_f1': 0.5986840157618951, 'f1_answerable': 0.8877705083039759, 'f1_unanswerable': 0.30959752321981426, 'accuracy': 0.8069264069264069, 'precision': 0.8855421686746988, 'recall': 0.8900100908173562, 'confusion_matrix': [[50, 114], [109, 882]]}


---
# Week 37:

## BIO Labeling of the training split

In [16]:
#CODE CELL WAS IMPORTED FROM lab2.ipynb
TOKEN_PATTERN = re.compile(r"\w+|[^\w\s]", re.UNICODE)

def tokenize_with_offsets(text):
    matches = list(TOKEN_PATTERN.finditer(text))
    tokens = [match.group(0) for match in matches]
    offsets = [(match.start(), match.end()) for match in matches]
    return tokens, offsets

def character_span_to_bio(context, answer_start=None, answer_text=""):
    tokens, offsets = tokenize_with_offsets(context)
    labels = ["O"] * len(tokens)

    if answer_start is None or answer_text == "":
        return tokens, offsets, labels

    answer_end = answer_start + len(answer_text)
    if context[answer_start:answer_end] != answer_text:
        raise ValueError("The supplied answer text does not match the character span")

    covered = [
        index
        for index, (start, end) in enumerate(offsets)
        if start < answer_end and end > answer_start
    ]
    if not covered:
        raise ValueError("The answer does not overlap any token")
    if offsets[covered[0]][0] != answer_start or offsets[covered[-1]][1] != answer_end:
        raise ValueError("The answer span does not align with token boundaries")

    labels[covered[0]] = "B-ANS"
    for index in covered[1:]:
        labels[index] = "I-ANS"
    return tokens, offsets, labels


def bio_to_character_span(context, offsets, labels):
    if len(offsets) != len(labels):
        raise ValueError("Offsets and labels must have the same length")
    if not set(labels) <= {"O", "B-ANS", "I-ANS"}:
        raise ValueError("Only O, B-ANS and I-ANS labels are supported")

    answer_indices = [
        index for index, label in enumerate(labels) if label != "O"
    ]
    if not answer_indices:
        return None, ""

    start_index = answer_indices[0]
    expected_indices = list(range(start_index, start_index + len(answer_indices)))
    expected_labels = ["B-ANS"] + ["I-ANS"] * (len(answer_indices) - 1)
    if answer_indices != expected_indices:
        raise ValueError("The labels contain multiple or non-contiguous answer spans")
    if [labels[index] for index in answer_indices] != expected_labels:
        raise ValueError("The answer span must begin with B-ANS and continue with I-ANS")

    start = offsets[answer_indices[0]][0]
    end = offsets[answer_indices[-1]][1]
    return start, context[start:end]

In [17]:
if checkpoint_exists("bio_labeling"):
    df, BIO_df, dropped_df = load_checkpoint("bio_labeling")
    print(f"Loaded cached BIO labeling: {len(BIO_df)} alignable rows, {len(dropped_df)} dropped rows")
else:
    df = df.copy()
    df = df.drop(columns=["tokens", "offsets", "labels", "alignable"], errors="ignore")

    def try_bio(row):
        if not row["answerable"]:
            # Unanswerable: always succeeds, labels are all "O"
            tokens, offsets, labels = character_span_to_bio(row["context"], None, "")
            return pd.Series({"tokens": tokens, "offsets": offsets, "labels": labels, "alignable": True})
        try:
            tokens, offsets, labels = character_span_to_bio(
                row["context"], row["answer_start"], row["answer"]
            )
            return pd.Series({"tokens": tokens, "offsets": offsets, "labels": labels, "alignable": True})
        except ValueError:
            # Answer text doesn't align to token boundaries we flag it.
            return pd.Series({"tokens": None, "offsets": None, "labels": None, "alignable": False})

    #Appending BIO columns to df
    bio_results = df.apply(try_bio, axis=1)
    df = pd.concat([df, bio_results], axis=1)

    print(f"{(~df['alignable']).sum()} / {len(df)} rows fail alignment")
    print(df.groupby("answerable")["alignable"].mean())

    # Derive the clean training view by filtering out the failing rows
    BIO_df = df[df["alignable"]].reset_index(drop=True)
    dropped_df = df[~df["alignable"]]  # kept around for reporting

    # Round-trip check: after BIO labeling, is the char length the same for tokens and labels?
    mismatches = 0
    for row in BIO_df.itertuples():
        if row.answerable:
            rt_start, rt_text = bio_to_character_span(row.context, row.offsets, row.labels)
            if rt_start != row.answer_start or rt_text != row.answer:
                mismatches += 1
        else:
            rt_start, rt_text = bio_to_character_span(row.context, row.offsets, row.labels)
            if rt_start is not None or rt_text != "":
                mismatches += 1

    print(f"{mismatches} / {len(BIO_df)} rows failed round-trip check")

    save_checkpoint("bio_labeling", (df, BIO_df, dropped_df))

BIO_df[["context", "answerable", "tokens", "labels"]]

84 / 7490 rows fail alignment
answerable
False    1.000000
True     0.987936
Name: alignable, dtype: float64
0 / 7406 rows failed round-trip check


,context,answerable,tokens,labels
0,The conflict between France and Spain continue...,True,"[The, conflict, between, France, and, Spain, c...","[O, O, O, B-ANS, O, O, O, O, O, O, O, O, O, O,..."
1,"X-rays make up X-radiation, a form of electrom...",True,"[X, -, rays, make, up, X, -, radiation, ,, a, ...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ..."
2,"In 2022, Beijing will become the first-ever ci...",True,"[In, 2022, ,, Beijing, will, become, the, firs...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ..."
3,The British Broadcasting Corporation (BBC) is ...,True,"[The, British, Broadcasting, Corporation, (, B...","[O, B-ANS, I-ANS, I-ANS, I-ANS, I-ANS, I-ANS, ..."
4,"Palestine ( '), officially the State of Palest...",True,"[Palestine, (, ', ), ,, officially, the, State...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ..."
...,...,...,...,...
7401,Reyyalagadda is a village belonging to Gangara...,True,"[Reyyalagadda, is, a, village, belonging, to, ...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ..."
7402,Boothumillipadu is a village in Gannavaram man...,True,"[Boothumillipadu, is, a, village, in, Gannavar...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ..."
7403,Mallavemula is a village belonging to Chagalam...,False,"[Mallavemula, is, a, village, belonging, to, C...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ..."
7404,"Andria Urushadze (; born April 25, 1968) is a ...",False,"[Andria, Urushadze, (, ;, born, April, 25, ,, ...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ..."


In [18]:
#A view of the rows where answerable = true but the length of the answer does not match
#with answer_start + len(answer)
dropped_df[["context", "answerable", "answer"]]

,context,answerable,answer
4861,The Australian radio-astronomer Dr John O'Sull...,True,Dr John O'Sullivan with his colleagues Terence...
4873,The Eye of Horus is an ancient Egyptian symbol...,True,Egypt
4899,British society has experienced significant ch...,True,social clas
4913,"The Battle of Leyte Gulf (Filipino: ""Labanan s...",True,The Battle of Leyte Gul
4920,Utilitarianism is an ethical and philosophical...,True,eremy Bentham
...,...,...,...
15474,"Internationally, IIT Delhi was ranked 172 in t...",True,IT Delhi
15701,These letters are believed to have derived fro...,True,Egypt
15901,"By early 2004, the show had settled into a reg...",True,"Keith Boak, Euros Lyn, Joe Ahearne, Brian Gran..."
16116,Early cultures identified celestial oFAncient ...,True,Ancient structures with possibly astronomical ...


In [19]:
def run_bio_checks():
    checks_passed = 0
    checks_total = 0

    def check(name, condition):
        nonlocal checks_passed, checks_total
        checks_total += 1
        status = "PASS" if condition else "FAIL"
        if condition:
            checks_passed += 1
        print(f"[{status}] {name}")

    #Answer starting at character 0
    context = "Copenhagen is the capital of Denmark."
    answer_text = "Copenhagen"
    answer_start = 0
    tokens, offsets, labels = character_span_to_bio(context, answer_start, answer_text)
    check("answer at char 0: first token is B-ANS", labels[0] == "B-ANS")
    check("answer at char 0: no other tokens are ANS", labels[1:] == ["O"] * (len(labels) - 1))
    rt_start, rt_text = bio_to_character_span(context, offsets, labels)
    check("answer at char 0: round-trips correctly", (rt_start, rt_text) == (answer_start, answer_text))

    #Multi-token answer
    context = "The main character of Jujutsu Kaisen is Itadori Yuji."
    answer_text = "Itadori Yuji"
    answer_start = context.index(answer_text)
    tokens, offsets, labels = character_span_to_bio(context, answer_start, answer_text)
    ans_indices = [i for i, l in enumerate(labels) if l != "O"]
    check("multi-token answer: spans exactly 2 tokens", len(ans_indices) == 2)
    check("multi-token answer: starts with B-ANS", labels[ans_indices[0]] == "B-ANS")
    check("multi-token answer: continues with I-ANS", all(labels[i] == "I-ANS" for i in ans_indices[1:]))
    rt_start, rt_text = bio_to_character_span(context, offsets, labels)
    check("multi-token answer: round-trips correctly", (rt_start, rt_text) == (answer_start, answer_text))

    #Punctuation immediately adjacent to the answer (no whitespace separator)
    context = "She was born in Odense, Denmark."
    answer_text = "Odense"
    answer_start = context.index(answer_text)  # immediately followed by ','
    tokens, offsets, labels = character_span_to_bio(context, answer_start, answer_text)
    ans_indices = [i for i, l in enumerate(labels) if l != "O"]
    check("punctuation-adjacent answer: exactly one ANS token (comma excluded)", len(ans_indices) == 1)
    check("punctuation-adjacent answer: comma token is 'O'", tokens[ans_indices[-1] + 1] == "," and labels[ans_indices[-1] + 1] == "O")
    rt_start, rt_text = bio_to_character_span(context, offsets, labels)
    check("punctuation-adjacent answer: round-trips correctly", (rt_start, rt_text) == (answer_start, answer_text))

    #Unanswerable example (no answer_start / answer_text)
    context = "This string contains no answer for the question."
    tokens, offsets, labels = character_span_to_bio(context, answer_start=None, answer_text="")
    check("unanswerable: all labels are 'O'", all(l == "O" for l in labels))
    check("unanswerable: tokens/offsets still populated", len(tokens) > 0 and len(offsets) == len(tokens))
    rt_start, rt_text = bio_to_character_span(context, offsets, labels)
    check("unanswerable: round-trips to (None, '')", (rt_start, rt_text) == (None, ""))

    print(f"\n{checks_passed}/{checks_total} checks passed")
    assert checks_passed == checks_total, "One or more BIO conversion checks failed"

run_bio_checks()

[PASS] answer at char 0: first token is B-ANS
[PASS] answer at char 0: no other tokens are ANS
[PASS] answer at char 0: round-trips correctly
[PASS] multi-token answer: spans exactly 2 tokens
[PASS] multi-token answer: starts with B-ANS
[PASS] multi-token answer: continues with I-ANS
[PASS] multi-token answer: round-trips correctly
[PASS] punctuation-adjacent answer: exactly one ANS token (comma excluded)
[PASS] punctuation-adjacent answer: comma token is 'O'
[PASS] punctuation-adjacent answer: round-trips correctly
[PASS] unanswerable: all labels are 'O'
[PASS] unanswerable: tokens/offsets still populated
[PASS] unanswerable: round-trips to (None, '')

13/13 checks passed


In [20]:
if checkpoint_exists("bio_df_translated"):
    BIO_df = load_checkpoint("bio_df_translated")
else:
    BIO_df["question_en"] = BIO_df.apply(lambda row: translate(row["question"], row["lang"]), axis=1)
    save_checkpoint("bio_df_translated", BIO_df)

In [21]:
#CODE CELL WAS IMPORTED FROM lab2.ipynb
def token_features(tokens, index):
    word = tokens[index]

    features = {
        "bias": 1.0,
        "word.lower": word.lower(),
        "word.prefix2": word[:2].lower(),
        "word.suffix2": word[-2:].lower(),
        "word.suffix3": word[-3:].lower(),
        "word.istitle": word.istitle(),
        "word.isupper": word.isupper(),
        "word.isdigit": word.isdigit(),
        "contains_hyphen": "-" in word,
    }

    if index == 0:
        features["BOS"] = True
    else:
        previous = tokens[index - 1]
        features.update({
            "previous.lower": previous.lower(),
            "previous.istitle": previous.istitle(),
            "previous.isupper": previous.isupper(),
        })

    if index == len(tokens) - 1:
        features["EOS"] = True
    else:
        following = tokens[index + 1]
        features.update({
            "next.lower": following.lower(),
            "next.istitle": following.istitle(),
            "next.isupper": following.isupper(),
        })

    return features

In [22]:
MONTH_NAMES = {"january", "february", "march", "april", "may", "june", "july",
    "august", "september", "october", "november", "december",}

def token_type_flags(token):
    return {
        "is_digit": token.isdigit(),
        "is_year_like": bool(re.fullmatch(r"(1|2)\d{3}", token)),
        "is_month_name": token.lower() in MONTH_NAMES,
        "is_capitalized_word": (token[:1].isupper() and token[1:].islower() if len(token) > 1 else token.isupper()),}

def qtype_match_feature(token, qtype):
    flags = token_type_flags(token)
    if qtype == "number":
        return flags["is_digit"]
    if qtype == "date":
        return flags["is_year_like"] or flags["is_month_name"] or flags["is_digit"]
    if qtype in ("person", "location"):
        return flags["is_capitalized_word"]
    return False


def token_features_qc(tokens, index, overlap_flags, qtype="other"):
    features = token_features(tokens, index)
    features["in_question"] = overlap_flags[index]
    features["prev_in_question"] = overlap_flags[index - 1] if index > 0 else False
    features["next_in_question"] = overlap_flags[index + 1] if index < len(tokens) - 1 else False

    window = 3
    lo, hi = max(0, index - window), min(len(tokens), index + window + 1)
    features["window_overlap_count"] = sum(overlap_flags[lo:hi])
    features["question_type"] = qtype
    features["qtype_match"] = qtype_match_feature(tokens[index], qtype)

    return features

In [23]:
from functools import lru_cache

@lru_cache(maxsize=200000)
def word_subword_pieces(word):
    pieces = tokenizer.tokenize(word.lower())
    return frozenset(p for p in pieces if p not in string.punctuation)

def compute_overlap_flags(tokens, q_subwords):
    if not q_subwords:
        return [False] * len(tokens)
    return [bool(word_subword_pieces(tok) & q_subwords) for tok in tokens]


In [24]:
def featurize_bio_df(df):
    features = []
    labels = []
    lengths = []

    for row in df.to_dict("records"):
        tokens = row["tokens"]
        question = row.get("question_en", row.get("question"))
        q_subwords = clean_tokens(question)
        overlap_flags = compute_overlap_flags(tokens, q_subwords)
        qtype = detect_question_type(question) #reuse question-type detector

        lengths.append(len(tokens))
        for i in range(len(tokens)):
            features.append(token_features_qc(tokens, i, overlap_flags, qtype))
        labels.extend(row["labels"])

    return features, labels, lengths


In [25]:
train_df_bio = BIO_df[BIO_df["split"] == "train"].copy()
val_df_bio = BIO_df[BIO_df["split"] == "validation"].copy()

In [26]:
if checkpoint_exists("bio_features"):
    train_features, train_labels, train_lengths, val_features, val_labels, val_lengths = load_checkpoint("bio_features")
else:
    train_features, train_labels, train_lengths = featurize_bio_df(train_df_bio)
    val_features, val_labels, val_lengths = featurize_bio_df(val_df_bio)
    save_checkpoint("bio_features", (train_features, train_labels, train_lengths, val_features, val_labels, val_lengths))

In [27]:
from sklearn.feature_extraction import DictVectorizer

if checkpoint_exists("qc_model"):
    vectorizer, qc_classifier = load_checkpoint("qc_model")
else:
    vectorizer = DictVectorizer(sparse=True)
    X_train = vectorizer.fit_transform(train_features)
    X_val = vectorizer.transform(val_features)

In [28]:
from sklearn.linear_model import SGDClassifier

if checkpoint_exists("qc_model"):
    print("Loaded cached vectorizer + qc_classifier, skipping fit.")
else:
    qc_classifier = SGDClassifier(
        loss="log_loss",
        alpha=1e-5,
        class_weight="balanced",
        max_iter=30,
        random_state=42)

    qc_classifier.fit(X_train, train_labels)
    save_checkpoint("qc_model", (vectorizer, qc_classifier))

In [29]:
def decode_best_span_greedy(predicted_labels):
    start_index = next((i for i, label in enumerate(predicted_labels) if label == "B-ANS"), None)
    if start_index is None:
        return None
    end_index = start_index
    while end_index + 1 < len(predicted_labels) and predicted_labels[end_index + 1] == "I-ANS":
        end_index += 1
    return start_index, end_index + 1


def decode_best_span_scored(probs, classes, max_span_len=25, eps=1e-12):
    b_idx = classes.index("B-ANS")
    i_idx = classes.index("I-ANS")
    o_idx = classes.index("O")

    n = len(probs)
    log_o = np.log(probs[:, o_idx] + eps)
    log_b = np.log(probs[:, b_idx] + eps)
    log_i = np.log(probs[:, i_idx] + eps)

    best_delta = 0.0
    best_span = None

    for start in range(n):
        running_delta = log_b[start] - log_o[start]
        if running_delta > best_delta:
            best_delta = running_delta
            best_span = (start, start + 1)
        end_limit = min(n, start + max_span_len)
        for end in range(start + 1, end_limit):
            running_delta += log_i[end] - log_o[end]
            if running_delta > best_delta:
                best_delta = running_delta
                best_span = (start, end + 1)

    return best_span

def span_to_bio_labels(n_tokens, span):
    labels = ["O"] * n_tokens
    if span is None:
        return labels
    start, end = span
    labels[start] = "B-ANS"
    for i in range(start + 1, end):
        labels[i] = "I-ANS"
    return labels


In [30]:
def predict_span_qc_classifier(row, vectorizer, classifier):
    tokens, offsets = row["tokens"], row["offsets"]
    question = row.get("question_en", row.get("question"))
    overlap_flags = compute_overlap_flags(tokens, clean_tokens(question))
    qtype = detect_question_type(question)

    features = [token_features_qc(tokens, i, overlap_flags, qtype) for i in range(len(tokens))]
    X = vectorizer.transform(features)
    probs = classifier.predict_proba(X)
    span = decode_best_span_scored(probs, list(classifier.classes_))

    if span is None:
        return None, ""
    start_index, end_index = span
    char_start, char_end = offsets[start_index][0], offsets[end_index - 1][1]
    return char_start, row["context"][char_start:char_end]


In [31]:
def empty_baseline(question, context):
    return None, ""

In [32]:
def predict_bio_labels(row, vectorizer, classifier):
    tokens = row["tokens"]
    question = row.get("question_en", row.get("question"))
    overlap_flags = compute_overlap_flags(tokens, clean_tokens(question))
    qtype = detect_question_type(question)

    features = [token_features_qc(tokens, i, overlap_flags, qtype) for i in range(len(tokens))]
    X = vectorizer.transform(features)

    probs = classifier.predict_proba(X)
    span = decode_best_span_scored(probs, list(classifier.classes_))

    return span_to_bio_labels(len(tokens), span)


In [33]:
def bio_spans(labels):
    spans = set()
    start = None
    for i, label in enumerate(labels + ["O"]):
        if label == "B-ANS":
            if start is not None:
                spans.add((start, i))
            start = i

        elif label == "I-ANS":
            if start is None:
                start = i

        else:
            if start is not None:
                spans.add((start, i))
                start = None

    return spans

In [34]:
from sklearn.metrics import precision_score, recall_score, f1_score

def token_level_scores(gold_sequences, predicted_sequences):
    gold_tokens, pred_tokens = [], []
    for gold, predicted in zip(gold_sequences, predicted_sequences):
        gold_tokens.extend([1 if l in {"B-ANS", "I-ANS"} else 0 for l in gold])
        pred_tokens.extend([1 if l in {"B-ANS", "I-ANS"} else 0 for l in predicted])
    return {
        "precision": precision_score(gold_tokens, pred_tokens, zero_division=0),
        "recall": recall_score(gold_tokens, pred_tokens, zero_division=0),
        "f1": f1_score(gold_tokens, pred_tokens, zero_division=0),
    }

def exact_span_match_rate(gold_sequences, predicted_sequences):
    matches = [bio_spans(g) == bio_spans(p) for g, p in zip(gold_sequences, predicted_sequences)]
    return sum(matches) / len(matches)

def answerability_agreement(gold_sequences, predicted_sequences):
    # Did the model correctly decide an answer exists (or doesn't), regardless of exact boundaries?
    correct = [bool(bio_spans(g)) == bool(bio_spans(p)) for g, p in zip(gold_sequences, predicted_sequences)]
    return sum(correct) / len(correct)


In [35]:
gold_bio_sequences = [row["labels"] for row in val_df_bio.to_dict("records")]
predicted_bio_sequences = [predict_bio_labels(row, vectorizer, qc_classifier) for row in val_df_bio.to_dict("records")]
empty_bio_sequences = [["O"] * len(g) for g in gold_bio_sequences]

qc_scores = token_level_scores(gold_bio_sequences, predicted_bio_sequences)
qc_exact = exact_span_match_rate(gold_bio_sequences, predicted_bio_sequences)
qc_answerability = answerability_agreement(gold_bio_sequences, predicted_bio_sequences)

empty_scores = token_level_scores(gold_bio_sequences, empty_bio_sequences)
empty_exact = exact_span_match_rate(gold_bio_sequences, empty_bio_sequences)
empty_answerability = answerability_agreement(gold_bio_sequences, empty_bio_sequences)

In [36]:
langs= val_df_bio["lang"].values
rows= []
for lang in ["All", "ar", "ko", "te"]:
    idx = np.arange(len(val_df_bio)) if lang == "All" else np.where(langs == lang)[0]
    gold = [gold_bio_sequences[i] for i in idx]
    n_unans = sum(1 for g in gold if not bio_spans(g))
    row = {"Lang": lang, "n": len(idx), "n_unans": n_unans}
    for name, preds in [("Empty", empty_bio_sequences), ("QC", predicted_bio_sequences)]:
        pred = [preds[i] for i in idx]
        s = token_level_scores(gold, pred)
        row[f"{name}_P"] = s["precision"]
        row[f"{name}_R"] = s["recall"]
        row[f"{name}_F1"] = s["f1"]
        row[f"{name}_Exact"] = exact_span_match_rate(gold, pred)
        row[f"{name}_Agree"] = answerability_agreement(gold, pred)
    rows.append(row)

span_table = pd.DataFrame(rows)
print(span_table.round(3).to_string(index=False))

Lang    n  n_unans  Empty_P  Empty_R  Empty_F1  Empty_Exact  Empty_Agree  QC_P  QC_R  QC_F1  QC_Exact  QC_Agree
 All 1150      164      0.0      0.0       0.0        0.143        0.143 0.318 0.125  0.180     0.176     0.766
  ar  415       52      0.0      0.0       0.0        0.125        0.125 0.322 0.114  0.168     0.190     0.788
  ko  352       19      0.0      0.0       0.0        0.054        0.054 0.374 0.124  0.186     0.176     0.832
  te  383       93      0.0      0.0       0.0        0.243        0.243 0.269 0.145  0.188     0.159     0.681


---
# Week 38:


In [37]:
from collections import Counter
import math

ko_df = df[df["lang"] == "ko"]
df_groups = ko_df.groupby("split")

PAD, UNK, BOS, EOS = "<pad>", "<unk>", "<bos>", "<eos>"

def subword_pieces(text):
  return tokenizer.tokenize(str(text).lower())

train_piece_counts = Counter()
for q in df_groups.get_group("train")["question"]:
  train_piece_counts.update(subword_pieces(q))

#Get the vocabulary from the training questions and string to index
# + index to string, so we can provide n word context to model and baseline
vocab = [PAD, UNK, BOS, EOS] + sorted(train_piece_counts)
stoi = {tok: i for i, tok in enumerate(vocab)}
itos = {i: tok for tok, i in stoi.items()}

def encode(text):
  ids = [stoi.get(p, stoi[UNK]) for p in subword_pieces(text)]
  return [stoi[BOS]] + ids + [stoi[EOS]]

train_texts = df_groups.get_group("train")["question"].tolist()
val_texts = df_groups.get_group("validation")["question"].tolist()

val_quest = df_groups.get_group("validation")["question"]
n_val_oov = sum(1 for q in val_quest for p in subword_pieces(q) if p not in stoi)
n_val_pieces = sum(len(subword_pieces(q)) for q in val_quest)
print(f"Vocav size in Korean training: {len(vocab)}")
print(f"Validation OOV rate: {n_val_oov}/{n_val_pieces} pieces "
      f"({100*n_val_oov/n_val_pieces:.2f}%) mapped to <unk>")

Vocav size in Korean training: 2227
Validation OOV rate: 114/4329 pieces (2.63%) mapped to <unk>


In [38]:
def corpus_perplexity(nll_fn, texts):
    total_nll, total_tok, per_q = 0.0, 0, []
    for t in texts:
        nll, n = nll_fn(t)
        total_nll += nll; total_tok += n
        per_q.append(math.exp(nll / n))
    return math.exp(total_nll / total_tok), np.array(per_q), total_tok

In [39]:
class AddAlphaNGram:
    def __init__(self, vocab, n, alpha):
        self.V = vocab
        self.n = n
        self.alpha = alpha
        self.c = defaultdict(Counter)
        self.h = Counter()

    def _seq(self, text):
        return [stoi[BOS]] * max(self.n - 2, 0) + encode(text)

    def _positions(self, ids):
        return range(max(self.n - 1, 1), len(ids))

    def fit(self, texts):
        for t in texts:
            ids = self._seq(t)
            for i in self._positions(ids):
                hist = tuple(ids[i - (self.n - 1):i])
                self.c[hist][ids[i]] += 1
                self.h[hist] += 1
        return self

    def prob(self, hist, w):
        counts = self.c.get(hist)
        cw = counts[w] if counts else 0
        return (cw + self.alpha) / (self.h.get(hist, 0) + self.alpha * len(self.V))

    def nll(self, text):
        ids = self._seq(text)
        total, k = 0.0, 0
        for i in self._positions(ids):
            hist = tuple(ids[i - (self.n - 1):i])
            total -= math.log(self.prob(hist, ids[i]))
            k += 1
        return total, k

## Baseline + Grid Seacrh n-gram (Heba)

In [40]:
#Baseline
unigram = AddAlphaNGram(vocab, 1, 1.0).fit(train_texts)
# Heba's model order and smoothing chosen on the validation questions
if checkpoint_exists("ngram_grid_ptok"):
    grid = load_checkpoint("ngram_grid_ptok")
else:
    grid = {}
    for n in (2, 3, 4):
        for alpha in (1.0, 0.1, 0.01, 0.001):
            m = AddAlphaNGram(vocab, n, alpha).fit(train_texts)
            grid[(n, alpha)] = corpus_perplexity(m.nll, val_texts)[0]
    save_checkpoint("ngram_grid_ptok", grid)

best_key = min(grid, key=grid.get)
ngram = AddAlphaNGram(vocab, *best_key).fit(train_texts)
print("best (n, alpha):", best_key)

best (n, alpha): (2, 0.01)


### Andreas Toftager Ejlersen (knm882) RNN Implementation:


In [41]:
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pad_sequence

device = "cuda" if torch.cuda.is_available() else "cpu"

class RNNLM(nn.Module):
    def __init__(self, vocab_size, emb_dim=128, hidden_dim=256):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=stoi[PAD])
        self.rnn = nn.RNN(emb_dim, hidden_dim, batch_first=True)
        self.out = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x):
        h, _ = self.rnn(self.emb(x))
        return self.out(h)

RNNmodel = RNNLM(len(vocab)).to(device)

In [42]:
def make_batch(texts, batch_size=32):
    ids = [torch.tensor(encode(t)) for t in texts]
    for i in range(0, len(ids), batch_size):
        yield pad_sequence(ids[i:i+batch_size], batch_first=True, padding_value=stoi[PAD])

#Training neural network via gradient descent
optimizer = torch.optim.Adam(RNNmodel.parameters(), lr=1e-3)
#we ignore padding tokens to not penalize the model for meaningless predictions.
criterion = nn.CrossEntropyLoss(ignore_index=stoi[PAD])

rnn_checkpoint_path = CACHE_DIR / "rnn_state.pt"
if rnn_checkpoint_path.exists():
    RNNmodel.load_state_dict(torch.load(rnn_checkpoint_path, map_location=device))
    print("Loaded trained RNN weights from checkpoint, skipping training.")
else:
    #running training 5 times with same questions.
    RNNmodel.train()
    for epoch in range(5):
        total_loss, n_batches = 0.0, 0
        for batch in make_batch(train_texts):
            batch = batch.to(device)
            #predict each token with the context of everything before.
            inputs, targets = batch[:, :-1], batch[:, 1:]
            logits = RNNmodel(inputs)
            loss = criterion(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            n_batches += 1
        print(f"epoch {epoch+1}: avg loss {total_loss/n_batches:.4f}")

    torch.save(RNNmodel.state_dict(), rnn_checkpoint_path)

epoch 1: avg loss 5.2041
epoch 2: avg loss 3.9942
epoch 3: avg loss 3.5432
epoch 4: avg loss 3.2110
epoch 5: avg loss 2.9386


In [43]:
@torch.no_grad()
def rnn_nll(text, model):
    model.eval()
    ids = torch.tensor(encode(text)).unsqueeze(0).to(device)
    inputs, targets = ids[:, :-1], ids[:, 1:]
    logits = model(inputs)
    loss = nn.functional.cross_entropy(
        logits.reshape(-1, logits.size(-1)),
        targets.reshape(-1),
        reduction="sum"
    )
    return loss.item(), targets.numel()


In [44]:
models = {
    "Unigram (add-1)": unigram.nll,
    f"{best_key[0]}-gram (add-{best_key[1]})": ngram.nll,
    "RNN": lambda text: rnn_nll(text, RNNmodel)}

rows, per_q = [], {}
for name, fn in models.items():
    val_ppl, pq, n_tok = corpus_perplexity(fn, val_texts)
    train_ppl = corpus_perplexity(fn, train_texts)[0]

    per_q[name] = pq
    rows.append({
        "model": name,
        "val ppl (per token)": val_ppl,
        "train ppl (per token)": train_ppl,
        "median per-question val ppl": np.median(pq),
        "val questions": len(val_texts),
        "val tokens": n_tok
    })

ppl_table = pd.DataFrame(rows).set_index("model").round(2)
display(ppl_table)

,val ppl (per token),train ppl (per token),median per-question val ppl,val questions,val tokens
model,,,,,
Unigram (add-1),321.88,287.76,304.65,356,4685
2-gram (add-0.01),43.80,12.50,38.81,356,4685
RNN,36.20,15.14,29.82,356,4685


In [45]:
def show_extremes(name, k=5):
    s = (pd.DataFrame({"q": val_texts, "ppl": per_q[name]}).drop_duplicates("q").sort_values("ppl"))
    for label, part in [("lowest", s.head(k)), ("highest", s.tail(k))]:
        print(f"\n{name} - {label} perplexity")
        for q, p in zip(part["q"], part["ppl"]):
            print(f"{p:7.1f}{q}->{translation_cache.get((q,"ko"))}")

for name in models:
    show_extremes(name)


Unigram (add-1) - lowest perplexity
   71.8세상에서 가장 큰 대학교는 무엇인가?->What is the largest university in the world?
   96.9일본에서 가장 높은 섬은 무엇인가?->What is the highest island in Japan?
   97.4세상에서 가장 오래된 대학교는 무엇인가?->What is the oldest university in the world?
  100.5로마의 수도는 어디인가?->Where is the capital of Rome?
  102.4중국에서 가장 작은 도시는 어디인가?->Where is the smallest city in China?

Unigram (add-1) - highest perplexity
  913.7더글러스 맥아더 배우자는 누군가요?->Who is Douglas MacArthur's wife?
  985.9논증의 오류로 인해 무지에의 호소 오류를 범할수 있는가?->Is it possible to make an unknown appeal because of an error in the argument?
 1156.0임시정부는 종전의 정부가 무너진 후, 무정부 상태를 해소하기 위해 임시로 구성된 정부를 말하나요?->Does the interim government say that it is a temporary government to dismantle the state of non-government after the collapse of the former government?
 1241.0탄광 노동자들이 파업을 한 이유는 무엇인가?->What was the reason for the strike?
 1345.7니콜라 테슬라와 Tesla, Inc.은 관계가 있는가?->Is there a connection between Nikola Tesla and Tesla, Inc.?

2-gram (add-0.01) - lowest per

In [47]:
def ngram_surprisals(m, text):
    ids = m._seq(text)
    rows = []
    for i in m._positions(ids):
        hist = tuple(ids[i - (m.n - 1): i])
        rows.append((itos[ids[i]], -math.log2(m.prob(hist, ids[i])), m.h[hist]))
    return rows

@torch.no_grad()
def rnn_surprisals(text):
    RNNmodel.eval()
    ids = torch.tensor(encode(text)).unsqueeze(0).to(device)
    lp = torch.log_softmax(RNNmodel(ids[:, :-1]), -1).gather(2, ids[:, 1:].unsqueeze(-1)).squeeze(-1)
    return [(itos[i], -l.item() / math.log(2)) for i, l in zip(ids[0, 1:].tolist(), lp[0])]

def compare_surprisals(q):
    a, b = ngram_surprisals(ngram, q), rnn_surprisals(q)
    print("\n", q, "->", translation_cache.get((q,"ko")))
    print(pd.DataFrame({"piece": [p for p, _, _ in a],
                        "ngram_bits": [round(x, 1) for _, x, _ in a],
                        "ctx_seen": [c for _, _, c in a],
                        "rnn_bits": [round(x, 1) for _, x in b]}).to_string(index=False))

ng_name = list(models)[1]
s = pd.Series(per_q[ng_name], index=val_texts)
s = s[~s.index.duplicated()]
compare_surprisals(s.idxmax())
compare_surprisals(s.idxmin())


 임시정부는 종전의 정부가 무너진 후, 무정부 상태를 해소하기 위해 임시로 구성된 정부를 말하나요? -> Does the interim government say that it is a temporary government to dismantle the state of non-government after the collapse of the former government?
piece  ngram_bits  ctx_seen  rnn_bits
   ▁임         9.3      2422       9.2
    시         4.9         8       5.6
   정부         6.0        43       9.2
    는        11.3         3       6.1
   ▁종        16.8      1154      15.6
    전        11.9        17       7.4
    의         6.2        50       2.5
<unk>        15.9       601      17.4
   ▁무        11.1         0      11.3
    너         2.8        27       4.2
    진        11.8        13       7.4
   ▁후        12.3        29      15.3
    ,        12.0        19      13.4
   ▁무        12.1        23      11.3
   정부        12.3        27      13.3
  ▁상태        11.3         3      12.7
    를        11.2         1       6.7
   ▁해        13.7       110      11.1
    소        12.4        33      13.9
   하기        12.8        48 

In [48]:
text = "니콜라 테슬라와 Tesla, Inc.은 관계가 있는가?"
pieces = subword_pieces(text)
for p in pieces:
    print(f"{p!r:20s} in vocab: {p in stoi}")

'▁니'                 in vocab: True
'콜라'                 in vocab: False
'▁테'                 in vocab: True
'슬'                  in vocab: True
'라'                  in vocab: True
'와'                  in vocab: True
'▁tes'               in vocab: False
'la'                 in vocab: False
','                  in vocab: True
'▁in'                in vocab: False
'c'                  in vocab: True
'.'                  in vocab: True
'은'                  in vocab: True
'▁관계'                in vocab: True
'가'                  in vocab: True
'▁있는'                in vocab: True
'가'                  in vocab: True
'?'                  in vocab: True


latin language tokens give 100% failure rate since they are not in the training set. The hypothesis that foreign language will tend to plummet performance seems to be true.

What is the "SOMETHING" in "COUNTY" is a question with low perplexity. Hypothesis is that due to short sentence length and frequent sighting of this sentence structure in training that the RNN predicts this type of sentence well.


Hypothesis: An n-gram language model assigns lower perplexity to text containing familiar local contexts that were frequently observed during training, while unfamiliar or unseen contexts lead to higher surprisal and therefore higher perplexity.

In [49]:
has_oov = np.array([any(p not in stoi for p in subword_pieces(t)) for t in val_texts])
for name, p in per_q.items():
    print(f"{name}: with OOV mean {p[has_oov].mean():.1f} / median {np.median(p[has_oov]):.1f} (n={has_oov.sum()}), "
          f"without {p[~has_oov].mean():.1f} / {np.median(p[~has_oov]):.1f} (n={(~has_oov).sum()})")

Unigram (add-1): with OOV mean 500.6 / median 455.1 (n=84), without 299.4 / 263.2 (n=272)
2-gram (add-0.01): with OOV mean 166.9 / median 108.7 (n=84), without 55.1 / 26.3 (n=272)
RNN: with OOV mean 152.9 / median 94.4 (n=84), without 45.3 / 22.3 (n=272)


---
# Week 39

In [50]:
from sklearn.metrics import precision_recall_fscore_support,accuracy_score

def evaluate_answerability(y_true,y_pred,model_name="model"):
    precision,recall,f1,support=precision_recall_fscore_support(y_true, y_pred,labels=[0, 1],zero_division=0)
    macro_f1=(f1[0]+f1[1])/2
    accuracy=accuracy_score(y_true, y_pred)

    result={
        "model": model_name,
        "macro_f1": macro_f1,
        "accuracy": accuracy,
        "unanswerable_precision": precision[0],
        "unanswerable_recall": recall[0],
        "unanswerable_f1": f1[0],
        "unanswerable_support": int(support[0]),
        "answerable_precision": precision[1],
        "answerable_recall": recall[1],
        "answerable_f1": f1[1],
        "answerable_support": int(support[1]),
    }

    return result

# Baseline Implementation

In [51]:
from scipy.sparse import hstack, csr_matrix
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

# Full df rather than BIO_df, so the rows dropped by BIO alignment are still classified
ans_train_df = df[df["split"] == "train"].copy()
ans_val_df = df[df["split"] == "validation"].copy()

def question_en_for(question, lang):
    en = translation_cache.get((question, lang))
    return en if en is not None else translate_cached(question, lang)

for frame in (ans_train_df, ans_val_df):
    frame["question_en"] = [question_en_for(q, l) or "" for q, l in zip(frame["question"], frame["lang"])]

DENSE_FEATURES = ["overlap_frac", "overlap_count", "log_q_len", "log_c_len", "log_len_ratio"]

def dense_features(frame):
    rows = []
    for q, q_en, c in zip(frame["question"], frame["question_en"], frame["context"]):
        # Overlap: translated question vs. (English) context, same tokens as lexical_overlap_score
        q_tok, c_tok = clean_tokens(q_en), clean_tokens(c)
        shared = len(q_tok & c_tok)
        overlap_frac = shared / len(q_tok) if q_tok else 0.0
        # Length: original-language question and context word counts
        q_len, c_len = n_tokens(q), n_tokens(c)
        rows.append([overlap_frac, shared, np.log1p(q_len), np.log1p(c_len),
                     np.log1p(c_len) - np.log1p(q_len)])
    return np.array(rows, dtype=float)

def build_features(frame, fit=False):
    if fit:
        X_q = q_vec.fit_transform(frame["question"])
        X_c = c_vec.fit_transform(frame["context"])
        X_d = scaler.fit_transform(dense_features(frame))
    else:
        X_q = q_vec.transform(frame["question"])
        X_c = c_vec.transform(frame["context"])
        X_d = scaler.transform(dense_features(frame))
    return hstack([X_q, X_c, csr_matrix(X_d)]).tocsr()

if checkpoint_exists("logreg_answerability"):
    q_vec, c_vec, scaler, logreg, X_ans_train, X_ans_val = load_checkpoint("logreg_answerability")
    print("Loaded cached LogReg baseline, skipping featurisation + fit.")
else:
    #Separate vectorizers so question and context words stay distinguishable in the weights
    bow_kwargs = dict(tokenizer=subword_pieces, lowercase=False, token_pattern=None, min_df=2)
    q_vec = CountVectorizer(**bow_kwargs)
    c_vec = CountVectorizer(**bow_kwargs)
    scaler = StandardScaler()

    X_ans_train = build_features(ans_train_df, fit=True)
    X_ans_val = build_features(ans_val_df)

    #94% of train is answerable, so balance the classes or it collapses to the majority baseline
    logreg = LogisticRegression(class_weight="balanced", max_iter=2000, random_state=42)
    logreg.fit(X_ans_train, ans_train_df["answerable"].astype(int).values)
    save_checkpoint("logreg_answerability", (q_vec, c_vec, scaler, logreg, X_ans_train, X_ans_val))

y_ans_train = ans_train_df["answerable"].astype(int).values
y_ans_val = ans_val_df["answerable"].astype(int).values
logreg_val_pred = logreg.predict(X_ans_val)

# Interpretability: dense feature weights + strongest BoW weights (positive = answerable)
feature_names = np.array(
    ["q:" + t for t in q_vec.get_feature_names_out()]
    + ["c:" + t for t in c_vec.get_feature_names_out()]
    + DENSE_FEATURES
)
coefs = logreg.coef_[0]
print(f"{X_ans_train.shape[1]} features ({len(q_vec.vocabulary_)} question BoW, "
      f"{len(c_vec.vocabulary_)} context BoW, {len(DENSE_FEATURES)} dense)\n")
print("Dense feature weights (standardised):")
for name, w in zip(DENSE_FEATURES, coefs[-len(DENSE_FEATURES):]):
    print(f"  {name:15s} {w:+.3f}")
order = np.argsort(coefs)
print("\nTop 10 -> answerable:  ", ", ".join(f"{feature_names[i]} ({coefs[i]:+.2f})" for i in order[::-1][:10]))
print("Top 10 -> unanswerable:", ", ".join(f"{feature_names[i]} ({coefs[i]:+.2f})" for i in order[:10]))

19722 features (5031 question BoW, 14686 context BoW, 5 dense)

Dense feature weights (standardised):
  overlap_frac    +0.041
  overlap_count   +0.202
  log_q_len       -0.240
  log_c_len       -0.132
  log_len_ratio   -0.028

Top 10 -> answerable:   q:▁무엇인가 (+1.02), q:▁ما (+1.01), q:تى (+0.97), q:▁언제 (+0.80), q:▁كم (+0.75), q:▁가장 (+0.68), q:▁몇 (+0.63), c:rabia (+0.52), q:▁어디 (+0.51), q:▁م (+0.50)
Top 10 -> unanswerable: q:▁هل (-6.27), c:light (-0.95), q:가 (-0.77), c:▁efficiency (-0.67), c:▁screen (-0.67), q:▁열 (-0.67), c:frica (-0.66), c:ally (-0.65), q:하는 (-0.65), c:ets (-0.62)


#BILSTM Classifier


In [52]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import confusion_matrix
import logging
from transformers import logging as tf_logging
tf_logging.set_verbosity_error()
logging.getLogger("transformers").setLevel(logging.ERROR)

torch.manual_seed(42)
np.random.seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-base")
xlmr = AutoModel.from_pretrained("xlm-roberta-base")
embeddings = xlmr.get_input_embeddings().weight.detach().clone()

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.12GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [53]:
def pool(x, m):
    return x.masked_fill(~m.unsqueeze(-1), -1e9).max(1).values


class BiLSTM(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb = nn.Embedding.from_pretrained(embeddings, freeze=True)
        self.lstm = nn.LSTM(768 + 2, 256, batch_first=True, bidirectional=True)
        self.drop = nn.Dropout(0.3)
        self.out = nn.Linear(4 * 512, 2)

    def forward(self, ids, mask):
        is_sep = (ids == 2).long()
        seg = ((is_sep.cumsum(1) - is_sep) > 0).long()

        q_ids = torch.where(mask & (seg == 0) & (ids > 2), ids, -1)
        c_ids = torch.where(mask & (seg == 1) & (ids > 2), ids, -1)
        in_q = (ids.unsqueeze(2) == q_ids.unsqueeze(1)).any(2)
        in_c = (ids.unsqueeze(2) == c_ids.unsqueeze(1)).any(2)
        match = torch.where(seg == 1, in_q, in_c) & (ids > 2)

        flags = torch.stack([seg.float(), match.float()], dim=-1)
        x = torch.cat([self.drop(self.emb(ids)), flags], dim=-1)

        packed = pack_padded_sequence(x, mask.sum(1).cpu(), batch_first=True, enforce_sorted=False)
        h, _ = self.lstm(packed)
        h, _ = pad_packed_sequence(h, batch_first=True, total_length=ids.size(1))

        q = pool(h, mask & (seg == 0))
        c = pool(h, mask & (seg == 1))
        return self.out(self.drop(torch.cat([q, c, q * c, (q - c).abs()], dim=-1)))


bilstm_model = BiLSTM().to(device)

In [54]:
def get_batch(frame, rows):
    part = frame.iloc[rows]
    enc = tokenizer(list(part["question"]), list(part["context"]),
                    truncation="only_second", max_length=384,
                    padding=True, return_tensors="pt")
    return enc["input_ids"].to(device), enc["attention_mask"].bool().to(device)


def predict(frame):
    bilstm_model.eval()
    preds = []
    with torch.no_grad():
        for start in range(0, len(frame), 64):
            rows = np.arange(start, min(start + 64, len(frame)))
            ids, mask = get_batch(frame, rows)
            preds.append(bilstm_model(ids, mask).argmax(-1).cpu().numpy())
    return np.concatenate(preds)


counts = np.bincount(y_ans_train)
weights = torch.tensor(len(y_ans_train) / (2 * counts), dtype=torch.float).to(device)
loss_fn = nn.CrossEntropyLoss(weight=weights)
optimizer = torch.optim.Adam([p for p in bilstm_model.parameters() if p.requires_grad], lr=0.001)

best_f1 = 0
for epoch in range(10):
    bilstm_model.train()
    order = np.random.permutation(len(ans_train_df))
    total_loss = 0
    for start in range(0, len(order), 32):
        rows = order[start:start + 32]
        ids, mask = get_batch(ans_train_df, rows)
        y = torch.tensor(y_ans_train[rows]).to(device)

        optimizer.zero_grad()
        loss = loss_fn(bilstm_model(ids, mask), y)
        loss.backward()
        nn.utils.clip_grad_norm_(bilstm_model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item() * len(rows)

    val_pred = predict(ans_val_df)
    r = evaluate_answerability(y_ans_val, val_pred, "BiLSTM")
    print(f"epoch {epoch + 1} | mean train loss {total_loss / len(order):.4f} | "
          f"val macro-F1 {r['macro_f1']:.4f} | unanswerable F1 {r['unanswerable_f1']:.3f} | "
          f"predicted unanswerable {(val_pred == 0).mean():.1%}")
    if r["macro_f1"] > best_f1:
        best_f1 = r["macro_f1"]
        torch.save(bilstm_model.state_dict(), "bilstm_best.pt")

bilstm_model.load_state_dict(torch.load("bilstm_best.pt"))
bilstm_val_pred = predict(ans_val_df)

epoch 1 | mean train loss 0.4763 | val macro-F1 0.7034 | unanswerable F1 0.464 | predicted unanswerable 5.2%
epoch 2 | mean train loss 0.3318 | val macro-F1 0.7490 | unanswerable F1 0.558 | predicted unanswerable 9.7%
epoch 3 | mean train loss 0.2539 | val macro-F1 0.7700 | unanswerable F1 0.598 | predicted unanswerable 11.0%
epoch 4 | mean train loss 0.2048 | val macro-F1 0.7745 | unanswerable F1 0.599 | predicted unanswerable 8.1%
epoch 5 | mean train loss 0.1704 | val macro-F1 0.8123 | unanswerable F1 0.679 | predicted unanswerable 14.6%
epoch 6 | mean train loss 0.1401 | val macro-F1 0.8386 | unanswerable F1 0.719 | predicted unanswerable 12.0%
epoch 7 | mean train loss 0.1011 | val macro-F1 0.8390 | unanswerable F1 0.725 | predicted unanswerable 14.7%
epoch 8 | mean train loss 0.0761 | val macro-F1 0.8395 | unanswerable F1 0.721 | predicted unanswerable 12.2%
epoch 9 | mean train loss 0.0619 | val macro-F1 0.8468 | unanswerable F1 0.736 | predicted unanswerable 13.3%
epoch 10 | me

## Models Evaluation


In [55]:
#RESULT OF LOGISTIC REGRESSION CLASSIFIER BASELINE
logreg_results = evaluate_answerability(y_ans_val, logreg_val_pred, "LogReg")
print(pd.Series(logreg_results).to_string())

print("\nMacro F1 per language:")
for lang in ["ar", "ko", "te"]:
    mask = (ans_val_df["lang"] == lang).values
    r = evaluate_answerability(y_ans_val[mask], logreg_val_pred[mask], f"LogReg ({lang})")
    print(f"  {lang}: macro F1 {r['macro_f1']:.3f}  unanswerable F1 {r['unanswerable_f1']:.3f}  accuracy {r['accuracy']:.3f}")

model                       LogReg
macro_f1                   0.71716
accuracy                  0.893506
unanswerable_precision    0.759494
unanswerable_recall       0.365854
unanswerable_f1           0.493827
unanswerable_support           164
answerable_precision      0.903346
answerable_recall         0.980827
answerable_f1             0.940493
answerable_support             991

Macro F1 per language:
  ar: macro F1 0.914  unanswerable F1 0.848  accuracy 0.964
  ko: macro F1 0.528  unanswerable F1 0.087  accuracy 0.941
  te: macro F1 0.573  unanswerable F1 0.281  accuracy 0.773


In [56]:
bilstm_results = evaluate_answerability(y_ans_val, bilstm_val_pred, "BILSTM")
print(pd.Series(bilstm_results).to_string())

print("\nMacro F1 per language:")
for lang in ["ar", "ko", "te"]:
    mask = (ans_val_df["lang"] == lang).values
    r = evaluate_answerability(y_ans_val[mask], bilstm_val_pred[mask], f"LogReg ({lang})")
    print(f"  {lang}: macro F1 {r['macro_f1']:.3f}  unanswerable F1 {r['unanswerable_f1']:.3f}  accuracy {r['accuracy']:.3f}")

model                       BILSTM
macro_f1                  0.856526
accuracy                  0.928139
unanswerable_precision    0.731429
unanswerable_recall       0.780488
unanswerable_f1           0.755162
unanswerable_support           164
answerable_precision      0.963265
answerable_recall         0.952573
answerable_f1             0.957889
answerable_support             991

Macro F1 per language:
  ar: macro F1 0.957  unanswerable F1 0.926  accuracy 0.981
  ko: macro F1 0.851  unanswerable F1 0.718  accuracy 0.969
  te: macro F1 0.778  unanswerable F1 0.667  accuracy 0.833


## Shared Evalution with 36 baselines

In [57]:
results = []
results.append(evaluate_answerability(y_ans_val,majority_val_pred,"Majority Baseline"))
results.append(evaluate_answerability(y_ans_val,final_pred,"Rule-Based"))
results.append(logreg_results)
results.append(bilstm_results)
results_df = pd.DataFrame(results).round(3)
results_df

,model,macro_f1,accuracy,unanswerable_precision,unanswerable_recall,unanswerable_f1,unanswerable_support,answerable_precision,answerable_recall,answerable_f1,answerable_support
0,Majority Baseline,0.462,0.858,0.000,0.000,0.000,164,0.858,1.000,0.924,991
1,Rule-Based,0.599,0.807,0.314,0.305,0.310,164,0.886,0.890,0.888,991
2,LogReg,0.717,0.894,0.759,0.366,0.494,164,0.903,0.981,0.940,991
3,BILSTM,0.857,0.928,0.731,0.780,0.755,164,0.963,0.953,0.958,991


In [58]:
model_predictions = {
    "Majority Baseline": majority_val_pred,
    "Rule-Based": final_pred,
    "Logistic Regression": logreg_val_pred,
    "BiLSTM": bilstm_val_pred,
}
lang_names = {"ar": "Ar", "ko": "Ko", "te": "Te"}
rows = []

for model_name, predictions in model_predictions.items():
    predictions = np.asarray(predictions).astype(int)
    for lang, lang_name in lang_names.items():
        mask = (ans_val_df["lang"] == lang).values
        r = evaluate_answerability(y_ans_val[mask], predictions[mask], model_name)
        rows.append({
            "Model": model_name,
            "Language": lang_name,
            "Macro F1": r["macro_f1"],
            "Unanswerable F1": r["unanswerable_f1"],
            "Unanswerable Precision": r["unanswerable_precision"],
            "Unanswerable Recall": r["unanswerable_recall"],
            "Answerable Precision": r["answerable_precision"],
            "Answerable Recall": r["answerable_recall"],
            "Answerable F1": r["answerable_f1"],
            "Accuracy": r["accuracy"],
        })

language_results_df=pd.DataFrame(rows).set_index(["Model", "Language"]).round(3)
language_results_df

Macro F1  Unanswerable F1  \
Model               Language                              
Majority Baseline   Ar           0.467            0.000   
                    Ko           0.486            0.000   
                    Te           0.431            0.000   
Rule-Based          Ar           0.572            0.239   
                    Ko           0.532            0.162   
                    Te           0.636            0.420   
Logistic Regression Ar           0.914            0.848   
                    Ko           0.528            0.087   
                    Te           0.573            0.281   
BiLSTM              Ar           0.957            0.926   
                    Ko           0.851            0.718   
                    Te           0.778            0.667   

                              Unanswerable Precision  Unanswerable Recall  \
Model               Language                                                
Majority Baseline   Ar                         0.000                0.000   
                    Ko                         0.000                0.000   
                    Te                         0.000                0.000   
Rule-Based          Ar                         0.275                0.212   
                    Ko                         0.109                0.316   
                    Te                         0.516                0.355   
Logistic Regression Ar                         0.894                0.808   
                    Ko                         0.250                0.053   
                    Te                         0.607                0.183   
BiLSTM              Ar                         0.893                0.962   
                    Ko                         0.700                0.737   
                    Te                         0.646                0.688   

                              Answerable Precision  Answerable Recall  \
Model               Language                                            
Majority Baseline   Ar                       0.875              1.000   
                    Ko                       0.947              1.000   
                    Te                       0.758              1.000   
Rule-Based          Ar                       0.891              0.920   
                    Ko                       0.957              0.855   
                    Te                       0.812              0.893   
Logistic Regression Ar                       0.973              0.986   
                    Ko                       0.949              0.991   
                    Te                       0.787              0.962   
BiLSTM              Ar                       0.994              0.983   
                    Ko                       0.985              0.982   
                    Te                       0.898              0.880   

                              Answerable F1  Accuracy  
Model               Language                           
Majority Baseline   Ar                0.933     0.875  
                    Ko                0.973     0.947  
                    Te                0.862     0.758  
Rule-Based          Ar                0.905     0.831  
                    Ko                0.903     0.826  
                    Te                0.851     0.763  
Logistic Regression Ar                0.979     0.964  
                    Ko                0.970     0.941  
                    Te                0.866     0.773  
BiLSTM              Ar                0.989     0.981  
                    Ko                0.984     0.969  
                    Te                0.889     0.833

## Inspection Part

In [59]:
ar_train = ans_train_df[ans_train_df["lang"] == "ar"]
is_yesno = ar_train["question"].str.strip().str.startswith("هل")
print(ar_train.groupby(is_yesno)["answerable"].agg(["mean", "count"]))

              mean  count
question                 
False     0.996414   2231
True      0.244648    327


From the baseline I found that Bag of words does the heavy lifting with regards to prediction of the label. in arabic, which performs much better than "ko" and "
"te", there is a specific character which basically determines that its unanswerable. thats why the baseline performs so much better on arabic, compared to "ko" and "te".

The biggest issue with the baseline is that the overwhelming majority of questions are answerable. Even when i try and make it balanced it is still really bad at predicting unanswerable q+c entries. I think this is the most important aspect for the LSTM to improve on.


In [60]:
ans_val_df["bilstm_pred"] = bilstm_val_pred
ans_val_df["true_label"] = y_ans_val
ans_val_df["error"] = np.where((ans_val_df["true_label"] == 0) & (ans_val_df["bilstm_pred"] == 1),"FP",
    np.where((ans_val_df["true_label"] == 1) & (ans_val_df["bilstm_pred"] == 0),"FN","Correct"))

errors = ans_val_df[ans_val_df["error"] != "Correct"]
print(pd.crosstab(errors["lang"], errors["error"]))
sample = errors.sample(frac=1, random_state=42).groupby(["lang", "error"]).head(3)

for i, r in sample.iterrows():
    print("=" * 90)
    print(f"row {i} | {r['lang']} | {r['error']}")
    print("Q :", r["question"])
    print("Q_en:", r["question_en"])
    print("CTX :", r["context"][:500])


error  FN  FP
lang         
ar      6   2
ko      6   5
te     35  29
row 15578 | te | FN
Q : భారత దేశంలో అధికంగా ఇస్లాం మతం ను ఏ రాష్ట్రంలో నమ్ముతారు?
Q_en: Which state in India has the largest Muslim population?
CTX : According to the Pew Research Center in 2010, there were 50 Muslim-majority countries. Around 62% of the world's Muslims live in the Asia-Pacific region (from Turkey to Indonesia), with over 1 billion adherents. The largest Muslim population in a country is in Indonesia, a nation home to 12.7% of the world's Muslims, followed by Pakistan (11.0%), and India (10.9%). About 20% of Muslims live in Arab countries. In the Middle East, the non-Arab countries of Iran and Turkey are the largest Muslim-ma
row 15343 | te | FN
Q : ఒరెగాన్ రాష్ట్రంలోని అతిపెద్ద నగరం ఏది ?
Q_en: What is the largest city in the state of Oregon?
CTX : Portland is the largest city in the U.S. state of Oregon and the seat of Multnomah County. It is a major port in the Willamette Valley region of the Paci

---
# Weeks 40-41: Integrated multilingual QA (XLM-R)

In [61]:
import collections, re as _re, string as _string, time
import torch
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForQuestionAnswering, TrainingArguments,
                          Trainer, default_data_collator, set_seed)

QA_MODEL = "xlm-roberta-base"      #same tokenizer family used earlier
MAX_LEN, STRIDE, MAX_ANS_LEN = 384, 128, 30
SEED = 42
QA_DIR = CACHE_DIR / f"qa_xlmr_seed{SEED}"
set_seed(SEED)

qa_tok = AutoTokenizer.from_pretrained(QA_MODEL)

QA_COLS = ["question", "context", "lang", "answerable", "answer_start", "answer"]
#Make it more modular, give a val or train split and fit to QA model
def to_qa_dataset(split):
    d = df[df["split"] == split][QA_COLS].reset_index(drop=True)
    d["answer_start"] = d["answer_start"].fillna(-1).astype(int)
    d["answer"] = d["answer"].fillna("")
    d["id"] = [f"{split}-{i}" for i in range(len(d))]
    return d

qa_train_df, qa_val_df = to_qa_dataset("train"), to_qa_dataset("validation")
qa_train_ds, qa_val_ds = Dataset.from_pandas(qa_train_df), Dataset.from_pandas(qa_val_df)


# --- Tokenisation: question = segment A, English context = segment B.
# Long contexts are split into overlapping windows; an unanswerable example, or a window
# that does not contain the answer, is labelled with start = end = 0 meaning the <s> token.
def prep_train(batch):
    enc = qa_tok(batch["question"], batch["context"], truncation="only_second",
                 max_length=MAX_LEN, stride=STRIDE, return_overflowing_tokens=True,
                 return_offsets_mapping=True, padding="max_length")
    sample_map = enc.pop("overflow_to_sample_mapping")
    starts, ends = [], []
    for i, offsets in enumerate(enc.pop("offset_mapping")):
        ex = sample_map[i]
        if not batch["answerable"][ex]:
            starts.append(0); ends.append(0); continue
        s_char = batch["answer_start"][ex]
        e_char = s_char + len(batch["answer"][ex])
        seq_ids = enc.sequence_ids(i)
        ctx_s = seq_ids.index(1)
        ctx_e = len(seq_ids) - 1 - seq_ids[::-1].index(1)
        if offsets[ctx_s][0] > s_char or offsets[ctx_e][1] < e_char:
            starts.append(0); ends.append(0); continue          # answer outside this window
        ts = ctx_s
        while ts <= ctx_e and offsets[ts][0] <= s_char: ts += 1
        te = ctx_e
        while te >= ctx_s and offsets[te][1] >= e_char: te -= 1
        starts.append(ts - 1); ends.append(te + 1)
    enc["start_positions"], enc["end_positions"] = starts, ends
    return enc

def prep_eval(batch):
    enc = qa_tok(batch["question"], batch["context"], truncation="only_second",
                 max_length=MAX_LEN, stride=STRIDE, return_overflowing_tokens=True,
                 return_offsets_mapping=True, padding="max_length")
    sample_map = enc.pop("overflow_to_sample_mapping")
    enc["example_id"] = [batch["id"][s] for s in sample_map]
    # keep offsets only for context tokens, so spans can never land in the question
    enc["offset_mapping"] = [
        [o if enc.sequence_ids(i)[k] == 1 else None for k, o in enumerate(offs)]
        for i, offs in enumerate(enc["offset_mapping"])
    ]
    return enc

train_feats = qa_train_ds.map(prep_train, batched=True, remove_columns=qa_train_ds.column_names)
val_feats = qa_val_ds.map(prep_eval, batched=True, remove_columns=qa_val_ds.column_names)
print(f"{len(qa_train_ds)} train examples -> {len(train_feats)} windows; "
      f"{len(qa_val_ds)} val examples -> {len(val_feats)} windows")


#Fine-tune one joint ar/ko/te model
if QA_DIR.exists():
    qa_model = AutoModelForQuestionAnswering.from_pretrained(QA_DIR)
    print("Loaded fine-tuned QA model from checkpoint, skipping training.")
else:
    qa_model = AutoModelForQuestionAnswering.from_pretrained(QA_MODEL)
    BATCH, EPOCHS = 16, 3
    total_steps = int(np.ceil(len(train_feats) / BATCH)) * EPOCHS
    args = TrainingArguments(
        output_dir=str(CACHE_DIR / "qa_runs"), per_device_train_batch_size=BATCH,
        learning_rate=3e-5, num_train_epochs=EPOCHS, weight_decay=0.01,
        warmup_steps=int(0.1 * total_steps),   # 10% linear warm-up
        fp16=torch.cuda.is_available(), seed=SEED, save_strategy="no", report_to="none",
    )
    trainer = Trainer(model=qa_model, args=args, train_dataset=train_feats,
                      data_collator=default_data_collator)
    t0 = time.time()
    trainer.train()
    print(f"Training time: {(time.time() - t0) / 60:.1f} min on "
          f"{torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
    trainer.save_model(QA_DIR)

pred_args = TrainingArguments(output_dir=str(CACHE_DIR / "qa_runs"), per_device_eval_batch_size=64,
                              report_to="none", fp16=torch.cuda.is_available())
qa_trainer = Trainer(model=qa_model, args=pred_args, data_collator=default_data_collator)
start_logits, end_logits = qa_trainer.predict(
    val_feats.remove_columns(["example_id", "offset_mapping"])).predictions


#best span + no-answer score per example
# no_answer_score = null (CLS) score - best non-empty span score.
# Higher score means more confident there is NO answer.
def postprocess(examples, feats, start_logits, end_logits, n_best=20):
    feat_ids = collections.defaultdict(list)
    for i, ex_id in enumerate(feats["example_id"]):
        feat_ids[ex_id].append(i)
    all_offsets = feats["offset_mapping"]
    out = {}
    for ex in examples:
        null_score, best_score, best_text, best_start = float("inf"), -float("inf"), "", -1
        for fi in feat_ids[ex["id"]]:
            s, e, offs = start_logits[fi], end_logits[fi], all_offsets[fi]
            null_score = min(null_score, s[0] + e[0])
            for a in np.argsort(s)[-n_best:]:
                for b in np.argsort(e)[-n_best:]:
                    if offs[a] is None or offs[b] is None or b < a or b - a + 1 > MAX_ANS_LEN:
                        continue
                    if s[a] + e[b] > best_score:
                        best_score = s[a] + e[b]
                        best_text = ex["context"][offs[a][0]:offs[b][1]]
                        best_start = offs[a][0]
        out[ex["id"]] = {"no_answer_score": float(null_score - best_score),
                         "text": best_text, "start": best_start}
    return out

qa_raw = postprocess(qa_val_ds, val_feats, start_logits, end_logits)


#SQuAD-v2 EM/F1 (official normalisation) + validation-tuned threshold
def _norm(s):
    s = "".join(ch for ch in s.lower() if ch not in set(_string.punctuation))
    s = _re.sub(r"\b(a|an|the)\b", " ", s)
    return " ".join(s.split())

def _f1(pred, gold):
    p, g = _norm(pred).split(), _norm(gold).split()
    if not p or not g:
        return float(p == g)
    common = sum((collections.Counter(p) & collections.Counter(g)).values())
    if common == 0:
        return 0.0
    prec, rec = common / len(p), common / len(g)
    return 2 * prec * rec / (prec + rec)

gold_ans = qa_val_df["answerable"].values.astype(bool)
gold_txt = qa_val_df["answer"].values
na_scores = np.array([qa_raw[i]["no_answer_score"] for i in qa_val_df["id"]])
span_txt = [qa_raw[i]["text"] for i in qa_val_df["id"]]
# per-example scores if the model answers (unanswerable + answering = 0)
em_if_answer = np.array([float(_norm(p) == _norm(g)) if a else 0.0
                         for p, g, a in zip(span_txt, gold_txt, gold_ans)])
f1_if_answer = np.array([_f1(p, g) if a else 0.0 for p, g, a in zip(span_txt, gold_txt, gold_ans)])

def squad_v2_scores(threshold, mask=slice(None)):
    abstain = na_scores[mask] > threshold
    correct_abstain = (~gold_ans[mask]).astype(float)
    em = np.where(abstain, correct_abstain, em_if_answer[mask]).mean()
    f1 = np.where(abstain, correct_abstain, f1_if_answer[mask]).mean()
    return em, f1

# tune the threshold on validation for SQuAD-v2 F1 (the project allows validation for thresholds)
candidates = np.concatenate([[-np.inf], np.sort(na_scores)])
qa_threshold = max(candidates, key=lambda t: squad_v2_scores(t)[1])
print(f"No-answer threshold: {qa_threshold:.3f} (abstain when no_answer_score > threshold)")


# --- Results overall and per language (always with n)
qa_pred_answerable = (na_scores <= qa_threshold).astype(int)
qa_rows = []
for lang in ["All", "ar", "ko", "te"]:
    m = np.ones(len(qa_val_df), bool) if lang == "All" else (qa_val_df["lang"] == lang).values
    em, f1 = squad_v2_scores(qa_threshold, m)
    ans = evaluate_answerability(gold_ans[m].astype(int), qa_pred_answerable[m], f"XLM-R ({lang})")
    qa_rows.append({"lang": lang, "n": int(m.sum()), "EM": em, "F1": f1,
                 "ans_macro_f1": ans["macro_f1"], "unans_P": ans["unanswerable_precision"],
                 "unans_R": ans["unanswerable_recall"], "unans_F1": ans["unanswerable_f1"]})
qa_results = pd.DataFrame(qa_rows)
print(qa_results.round(3).to_string(index=False))


#Abstention vs. Week 36/39 classifiers (same validation rows, same evaluation code)
assert (qa_val_df["question"].values == ans_val_df["question"].values).all(), "row order differs"
model_predictions["XLM-R QA"] = qa_pred_answerable
qa_compare = []
for name, preds in model_predictions.items():
    preds = np.asarray(preds).astype(int)
    for lang in ["All", "ar", "ko", "te"]:
        m = np.ones(len(ans_val_df), bool) if lang == "All" else (ans_val_df["lang"] == lang).values
        r = evaluate_answerability(y_ans_val[m], preds[m], name)
        qa_compare.append({"Model": name, "Language": lang, "n": int(m.sum()),
                           "Macro F1": r["macro_f1"], "Unans. F1": r["unanswerable_f1"],
                           "Unans. P": r["unanswerable_precision"], "Unans. R": r["unanswerable_recall"]})
abstention_df = pd.DataFrame(qa_compare).set_index(["Model", "Language"]).round(3)
print(abstention_df.to_string())

#Spans vs. the Week 37 sequence labeller, on the same BIO-alignable validation rows
# Exact span match as in Week 37: an unanswerable example counts as correct iff the model abstains;
# an answerable one iff the predicted character span equals the gold span.
qa_by_pair = {(q, c): (i, a) for i, q, c, a in zip(qa_val_df["id"], qa_val_df["question"],
                                                   qa_val_df["context"], qa_pred_answerable)}
span_hits = []
for row in val_df_bio.itertuples():
    ex_id, pred_ans = qa_by_pair[(row.question, row.context)]
    if not row.answerable:
        span_hits.append(pred_ans == 0)
    else:
        p = qa_raw[ex_id]
        span_hits.append(bool(pred_ans) and p["start"] == row.answer_start and p["text"] == row.answer)
print(f"\nExact span match on {len(val_df_bio)} BIO-alignable validation rows: "
      f"XLM-R {np.mean(span_hits):.3f} | Week 37 QC labeller {qc_exact:.3f} | empty baseline {empty_exact:.3f}")

Map:   0%|          | 0/6335 [00:00<?, ? examples/s]

Map:   0%|          | 0/1155 [00:00<?, ? examples/s]

6335 train examples -> 6562 windows; 1155 val examples -> 1194 windows


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

{'loss': '2.757', 'grad_norm': '26.3', 'learning_rate': '1.984e-05', 'epoch': '1.217'}
{'loss': '1.391', 'grad_norm': '23.06', 'learning_rate': '6.324e-06', 'epoch': '2.433'}
{'train_runtime': '395', 'train_samples_per_second': '49.84', 'train_steps_per_second': '3.122', 'train_loss': '1.888', 'epoch': '3'}
Training time: 6.6 min on Tesla T4


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

No-answer threshold: -4.593 (abstain when no_answer_score > threshold)
lang    n    EM    F1  ans_macro_f1  unans_P  unans_R  unans_F1
 All 1155 0.551 0.646         0.850    0.703    0.793     0.745
  ar  415 0.559 0.654         0.898    0.725    0.962     0.826
  ko  356 0.548 0.664         0.829    0.571    0.842     0.681
  te  384 0.544 0.620         0.808    0.727    0.688     0.707
                                 n  Macro F1  Unans. F1  Unans. P  Unans. R
Model               Language                                               
Majority Baseline   All       1155     0.462      0.000     0.000     0.000
                    ar         415     0.467      0.000     0.000     0.000
                    ko         356     0.486      0.000     0.000     0.000
                    te         384     0.431      0.000     0.000     0.000
Rule-Based          All       1155     0.599      0.310     0.314     0.305
                    ar         415     0.572      0.239     0.275     0.212
 

XLM-R matches the best classifier overall, is the strongest on Telugu, and is the only model that extracts spans well. Its weaker Arabic and Korean scores mostly reflect the shared threshold and the absence of the هل shortcut, not a weaker model.

In [62]:
import numpy as np, pandas as pd

def auto_category(gold_ans, pred_ans, gs, ge, ps, pe):
    if not gold_ans and pred_ans:
        return "missed abstention"
    if gold_ans and not pred_ans:
        return "false abstention"
    if ps <= gs and pe >= ge:
        return "over-extension (pred contains gold)"
    if ps >= gs and pe <= ge:
        return "under-extension (pred inside gold)"
    if min(pe, ge) - max(ps, gs) > 0:
        return "shifted boundary (partial overlap)"
    return "disjoint (different span)"

rows = []
for i, r in qa_val_df.reset_index(drop=True).iterrows():
    raw = qa_raw[r["id"]]
    gold_ans = bool(r["answerable"])
    pred_ans = bool(qa_pred_answerable[i])
    pred_text = raw["text"] if pred_ans else ""
    gold_text = r["answer"] if gold_ans else ""

    # same logic as squad_v2_scores unanswerable is correct iff the model abstains
    correct = (not pred_ans) if not gold_ans else (pred_ans and _norm(pred_text) == _norm(gold_text))
    if correct:
        continue

    gs, ge = (int(r["answer_start"]), int(r["answer_start"]) + len(gold_text)) if gold_ans else (-1, -1)
    ps, pe = (raw["start"], raw["start"] + len(pred_text)) if pred_ans else (-1, -1)
    rows.append({
        "id": r["id"], "lang": r["lang"], "question": r["question"], "context": r["context"],
        "gold": gold_text, "pred": pred_text,
        "gs": gs, "ge": ge, "ps": ps, "pe": pe,
        "auto_category": auto_category(gold_ans, pred_ans, gs, ge, ps, pe),
        "token_f1": _f1(pred_text, gold_text) if gold_ans and pred_ans else 0.0,
        "no_answer_score": raw["no_answer_score"],
    })

qa_errors_df = pd.DataFrame(rows)
print("Total errors:", len(qa_errors_df))
print(pd.crosstab(qa_errors_df["lang"], qa_errors_df["auto_category"], margins=True))
print("\nMean token F1 of errors by category:")
print(qa_errors_df.groupby("auto_category")["token_f1"].agg(["count", "mean"]).round(3))

def show_window(ctx, spans, pad=150):
    spans = [s for s in spans if s[0] >= 0]
    if not spans:
        return ctx[:400]
    lo, hi = min(s[0] for s in spans), max(s[1] for s in spans)
    return ("…" if lo > pad else "") + ctx[max(0, lo - pad):hi + pad] + ("…" if hi + pad < len(ctx) else "")

# random, stratified sample up to 3 errors per (language, category)
shuffled = qa_errors_df.sample(frac=1, random_state=0)
sample = shuffled.groupby(["lang", "auto_category"]).head(3).sort_values(["lang", "auto_category"])

for _, r in sample.iterrows():
    print("-" * 70)
    print(f'{r["id"]} [{r["lang"]}] {r["auto_category"]}  (F1={r["token_f1"]:.2f}, na_score={r["no_answer_score"]:.2f})')
    print("Q:", r["question"])
    print("Context:", show_window(r["context"], [(r["gs"], r["ge"]), (r["ps"], r["pe"])]))
    print("Gold:", repr(r["gold"]), "| Pred:", repr(r["pred"]))

#Week 39 comparison: answerability errors of each model, same validation rows
gold_a = y_ans_val.astype(int)
xlmr = np.asarray(model_predictions["XLM-R QA"]).astype(int)
for name, preds in model_predictions.items():
    if name == "XLM-R QA":
        continue
    p = np.asarray(preds).astype(int)
    print(f"\nXLM-R vs {name}: both right {((xlmr==gold_a)&(p==gold_a)).sum()}, "
          f"XLM-R only right {((xlmr==gold_a)&(p!=gold_a)).sum()}, "
          f"{name} only right {((xlmr!=gold_a)&(p==gold_a)).sum()}, "
          f"both wrong {((xlmr!=gold_a)&(p!=gold_a)).sum()}")

Total errors: 519
auto_category  disjoint (different span)  false abstention  missed abstention  \
lang                                                                            
ar                                   102                19                  2   
ko                                    78                12                  3   
te                                    59                24                 29   
All                                  239                55                 34   

auto_category  over-extension (pred contains gold)  \
lang                                                 
ar                                              34   
ko                                              46   
te                                              34   
All                                            114   

auto_category  shifted boundary (partial overlap)  \
lang                                                
ar                                              2   
ko         

#Week 43: Personal Extensions

## knm882: Conditioned end prediction of answer window

"If boundary errors come from independent start/end prediction, a start-conditioned end head will reduce the boundary-mismatch rate and raise EM on answerable examples, with a smaller effect on F1."

In [63]:
import torch.nn as nn, torch.nn.functional as F
from transformers import AutoModel

class StartConditionedQA(nn.Module):
    def __init__(self, name=QA_MODEL, k=20):              # k = 20 matches core n_best
        super().__init__()
        self.enc = AutoModel.from_pretrained(name)
        h = self.enc.config.hidden_size
        self.start = nn.Linear(h, 1)
        self.end = nn.Sequential(nn.Linear(2 * h, h), nn.Tanh(), nn.LayerNorm(h), nn.Linear(h, 1))
        self.k = k

    def end_given(self, H, i):                              # end logits conditioned on start i
        h_i = H[torch.arange(len(H), device=H.device), i].unsqueeze(1).expand_as(H)
        return self.end(torch.cat([H, h_i], -1)).squeeze(-1)

    def forward(self, input_ids, attention_mask, start_positions=None, end_positions=None):
        H = self.enc(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        s = self.start(H).squeeze(-1)
        if start_positions is not None:                     # training: gold start (teacher forcing)
            e = self.end_given(H, start_positions)
            return {"loss": (F.cross_entropy(s, start_positions) + F.cross_entropy(e, end_positions)) / 2}
        top = s.topk(self.k, -1).indices                    # inference: ends for top-k starts
        end_k = torch.stack([self.end_given(H, top[:, j]) for j in range(self.k)], 1)
        null_end = self.end_given(H, torch.zeros_like(top[:, 0]))
        # log-probabilities, so scores from different conditional end distributions are comparable
        return {"start_logits": F.log_softmax(s.float(), -1), "top": top,
                "end_k": F.log_softmax(end_k.float(), -1),
                "null_end": F.log_softmax(null_end.float(), -1)[:, 0]}

In [64]:
def train_or_load(kind, seed):                              # kind: "xlmr" (core) or "sc"
    d = CACHE_DIR / f"qa_{kind}_seed{seed}"                 # xlmr seed 42 == existing QA_DIR
    if kind == "xlmr" and d.exists():
        return AutoModelForQuestionAnswering.from_pretrained(d)
    if kind == "sc" and (d / "model.pt").exists():
        m = StartConditionedQA(); m.load_state_dict(torch.load(d / "model.pt")); return m
    set_seed(seed)
    m = AutoModelForQuestionAnswering.from_pretrained(QA_MODEL) if kind == "xlmr" else StartConditionedQA()
    steps = int(np.ceil(len(train_feats) / 16)) * 3
    args = TrainingArguments(output_dir=str(CACHE_DIR / "qa_runs"), per_device_train_batch_size=16,
                             learning_rate=3e-5, num_train_epochs=3, weight_decay=0.01,
                             warmup_steps=int(0.1 * steps), fp16=torch.cuda.is_available(),
                             seed=seed, save_strategy="no", report_to="none")
    t0 = time.time()
    Trainer(model=m, args=args, train_dataset=train_feats, data_collator=default_data_collator).train()
    print(f"{kind} seed {seed}: {(time.time() - t0) / 60:.1f} min")
    if kind == "xlmr":
        m.save_pretrained(d)
    else:
        d.mkdir(parents=True, exist_ok=True); torch.save(m.state_dict(), d / "model.pt")
    return m

def predict_raw(m, kind):
    out = Trainer(model=m, args=pred_args, data_collator=default_data_collator).predict(
        val_feats.remove_columns(["example_id", "offset_mapping"])).predictions
    return (postprocess if kind == "xlmr" else postprocess_sc)(qa_val_ds, val_feats, *out)

def postprocess_sc(examples, feats, S, TOP, END_K, NULL_END, n_best=20):
    # same constraints and output format as postprocess(), but ends depend on the start
    ids = collections.defaultdict(list)
    for i, ex_id in enumerate(feats["example_id"]):
        ids[ex_id].append(i)
    offsets, out = feats["offset_mapping"], {}
    for ex in examples:
        null, best, text, start = float("inf"), -float("inf"), "", -1
        for fi in ids[ex["id"]]:
            offs = offsets[fi]
            null = min(null, S[fi][0] + NULL_END[fi])
            for j, a in enumerate(TOP[fi]):
                for b in np.argsort(END_K[fi, j])[-n_best:]:
                    if offs[a] is None or offs[b] is None or b < a or b - a + 1 > MAX_ANS_LEN:
                        continue
                    if S[fi][a] + END_K[fi, j][b] > best:
                        best, text, start = S[fi][a] + END_K[fi, j][b], ex["context"][offs[a][0]:offs[b][1]], offs[a][0]
        out[ex["id"]] = {"no_answer_score": float(null - best), "text": text, "start": start}
    return out

In [65]:
def evaluate(raw):
    ids = qa_val_df["id"]
    ga  = qa_val_df["answerable"].values.astype(bool)   # need local var instead of global var.
    gt  = qa_val_df["answer"].values
    na  = np.array([raw[i]["no_answer_score"] for i in ids])
    txt = [raw[i]["text"] for i in ids]
    em  = np.array([float(_norm(p) == _norm(g)) for p, g in zip(txt, gt)]) * ga
    f1  = np.array([_f1(p, g) for p, g in zip(txt, gt)]) * ga
    ps = np.array([raw[i]["start"] for i in ids]); pe = ps + np.array([len(t) for t in txt])
    gs = qa_val_df["answer_start"].values;        ge = gs + np.array([len(g) for g in gt])
    wrong, overlap = ga & (em == 0), (ps >= 0) & (ps < ge) & (gs < pe)

    def sq(t, m):
        ab, ok = na[m] > t, (~ga[m]).astype(float)
        return np.where(ab, ok, em[m]).mean(), np.where(ab, ok, f1[m]).mean()

    full = np.ones(len(ids), bool)
    thr = max(np.concatenate([[-np.inf], np.sort(na)]), key=lambda t: sq(t, full)[1])
    pred = (na <= thr).astype(int)
    rows = []
    for lang in ["All", "ar", "ko", "te"]:
        m = full if lang == "All" else (qa_val_df["lang"] == lang).values
        a = m & ga
        rows.append({"lang": lang, "n": int(m.sum()), **dict(zip(["EM", "F1"], sq(thr, m))),
                     "ans_macro_f1": evaluate_answerability(ga[m].astype(int), pred[m], lang)["macro_f1"],
                     "ans_EM": em[a].mean(), "ans_F1": f1[a].mean(),
                     "boundary": int((wrong & overlap & m).sum()), "disjoint": int((wrong & ~overlap & m).sum())})
    return pd.DataFrame(rows), thr, pred

In [66]:
rows = []
for seed in [42, 43, 44]:
    for kind in ["xlmr", "sc"]:
        m = train_or_load(kind, seed)
        tab, thr, pred = evaluate(predict_raw(m, kind))
        rows.append(tab.assign(model=kind, seed=seed, threshold=thr))
        if (kind, seed) == ("sc", 42):
            model_predictions["XLM-R start-cond."] = pred   # frozen for the challenge set
        del m; torch.cuda.empty_cache()

res = pd.concat(rows)
res["boundary_share"] = res["boundary"] / (res["boundary"] + res["disjoint"])
cols = ["EM", "F1", "ans_macro_f1", "ans_EM", "ans_F1", "boundary", "disjoint", "boundary_share"]
print(res.groupby(["model", "lang"])[cols].agg(["mean", "std"]).round(3).to_string())

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

{'loss': '2.641', 'grad_norm': '15.04', 'learning_rate': '1.984e-05', 'epoch': '1.217'}
{'loss': '1.223', 'grad_norm': '20.9', 'learning_rate': '6.324e-06', 'epoch': '2.433'}
{'train_runtime': '393.9', 'train_samples_per_second': '49.97', 'train_steps_per_second': '3.13', 'train_loss': '1.746', 'epoch': '3'}
sc seed 42: 6.6 min


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

{'loss': '2.669', 'grad_norm': '34.55', 'learning_rate': '1.984e-05', 'epoch': '1.217'}
{'loss': '1.342', 'grad_norm': '20.13', 'learning_rate': '6.324e-06', 'epoch': '2.433'}
{'train_runtime': '391.9', 'train_samples_per_second': '50.23', 'train_steps_per_second': '3.146', 'train_loss': '1.833', 'epoch': '3'}
xlmr seed 43: 6.6 min


...

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

{'loss': '2.839', 'grad_norm': '31.54', 'learning_rate': '1.984e-05', 'epoch': '1.217'}
{'loss': '1.402', 'grad_norm': '36.63', 'learning_rate': '6.324e-06', 'epoch': '2.433'}
{'train_runtime': '414.6', 'train_samples_per_second': '47.48', 'train_steps_per_second': '2.974', 'train_loss': '1.936', 'epoch': '3'}
xlmr seed 44: 7.0 min


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

{'loss': '2.697', 'grad_norm': '19.38', 'learning_rate': '1.984e-05', 'epoch': '1.217'}
{'loss': '1.229', 'grad_norm': '25.1', 'learning_rate': '6.324e-06', 'epoch': '2.433'}
{'train_runtime': '399.9', 'train_samples_per_second': '49.23', 'train_steps_per_second': '3.083', 'train_loss': '1.778', 'epoch': '3'}
sc seed 44: 6.7 min
               EM            F1        ans_macro_f1        ans_EM        ans_F1        boundary         disjoint         boundary_share       
             mean    std   mean    std         mean    std   mean    std   mean    std     mean     std     mean     std           mean    std
model lang                                                                                                                                    
sc    All   0.525  0.013  0.604  0.015        0.771  0.006  0.527  0.015  0.625  0.018  159.333  15.275  309.333  25.891          0.341  0.040
      ar    0.559  0.021  0.639  0.022        0.896  0.017  0.507  0.017  0.607  0.019   55.333  

In [67]:
for seed in [42, 43, 44]:
    m = train_or_load("sc", seed)                       # loads checkpoint, no training
    tab, thr, pred = evaluate(predict_raw(m, "sc"))
    rows.append(tab.assign(model="sc_logprob", seed=seed, threshold=thr))
    if seed == 42:
        model_predictions["XLM-R start-cond."] = pred
    del m; torch.cuda.empty_cache()

res = pd.concat(rows)
res["boundary_share"] = res["boundary"] / (res["boundary"] + res["disjoint"])
print(res.groupby(["model", "lang"])[cols].agg(["mean", "std"]).round(3).to_string())

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

                    EM            F1        ans_macro_f1        ans_EM        ans_F1        boundary         disjoint         boundary_share       
                  mean    std   mean    std         mean    std   mean    std   mean    std     mean     std     mean     std           mean    std
model      lang                                                                                                                                    
sc         All   0.525  0.013  0.604  0.015        0.771  0.006  0.527  0.015  0.625  0.018  159.333  15.275  309.333  25.891          0.341  0.040
           ar    0.559  0.021  0.639  0.022        0.896  0.017  0.507  0.017  0.607  0.019   55.333   6.028  123.667   9.292          0.309  0.038
           ko    0.567  0.007  0.647  0.019        0.784  0.024  0.562  0.007  0.653  0.017   48.667   7.638   99.000   9.849          0.330  0.057
           te    0.449  0.012  0.525  0.012        0.635  0.018  0.512  0.024  0.614  0.024   55.333   2.517   8

### Outcome: hypothesis refuted

With log-probability decoding, the start-conditioned head does not improve span extraction. Answerable EM is unchanged (0.528 vs. 0.529, within the core model's seed std of 0.010), so the primary criterion is not met. The boundary share fell (0.43 → 0.36), but only because the errors shifted: boundary errors decreased (199 → 166) while disjoint errors increased by about the same amount (268 → 302), contradicting prediction 3. Answerable F1 (0.647 → 0.630) and answerability macro-F1 (0.810 → 0.764) both decreased. The pattern is the same in all three languages.

**Interpretation.** Conditioning the end on the start couples their errors. With independent heads, a wrong start is often rescued by a correct end, yielding an overlapping span with partial F1 credit (a boundary error). A conditioned end head places the end relative to whatever start it is given, so a wrong start displaces the whole span into the wrong region (a disjoint error). The independent end head therefore acts as a second, independent vote on the answer's location, and conditioning removes it. This is consistent with EM being unchanged while F1 decreases: near-misses that earned partial credit become complete misses. Exposure bias from teacher forcing (the end head never saw wrong starts during training) likely amplifies the effect.

**SIDE NOTE** The first decoding variant summed raw logits across separately normalised conditional distributions, which made scores incomparable across starts (answerable EM 0.417, 417 disjoint errors). Switching to log-probabilities was the only post change. Both variants are reported.

---
## Translating questions to English before applying an English QA model

Translating the Arabic, Korean and Telugu questions into English (NLLB) and fine-tuning an English extractive QA model (roberta-base) on them with the English contexts gives a higher SQuAD-v2 token-level F1 on the full validation set
(our primary metric, as in the shared evaluation) than fine-tuning the
multilingual XLM-R-base on the original-language questions. Training data, hyperparameters, and the answerable validation examples are the same for both.

In [68]:
def add_en(frame):
    f = frame.copy()
    f["question_orig"]=f["question"]
    f["question"]=[translation_cache.get((q, l)) or "" for q, l in zip(f["question_orig"], f["lang"])]
    return f

qa_train_en_df,qa_val_en_df= add_en(qa_train_df),add_en(qa_val_df)
print("empty translations(train/val):",(qa_train_en_df["question"] == "").sum(),(qa_val_en_df["question"] == "").sum())
print(qa_val_en_df[["lang", "question_orig", "question"]].sample(6, random_state=42).to_string())

empty translations(train/val): 0 0
    lang                                question_orig                                               question
964   ar             ما معنى كلمة الهاجاناه بالعبرية؟            What does the word hajjanah mean in Hebrew?
265   te             ఇంగ్లాండ్‍ దేశ జాతీయ క్రీడ ఏది ?                 What is the national sport of England?
109   te  2011లో అమెరికా అధ్యక్షుడిగా ఎవరు పనిచేసారు?  Who served as President of the United States in 2011?
299   ko                          일본에서 가장 높은 섬은 무엇인가?                   What is the highest island in Japan?
844   ar     هل الهرمونات هى المسبب الرئيسي للعواطف ؟              Are hormones the main cause of emotions ?
170   te        ప్రోగ్రామింగ్ భాషను ఎవరు కనిపెట్టారు?                 Who invented the programming language?


In [69]:
SQ= "roberta-base"
sq_tok= AutoTokenizer.from_pretrained(SQ)
qa_train_en_ds= Dataset.from_pandas(qa_train_en_df.reset_index(drop=True))
qa_val_en_ds= Dataset.from_pandas(qa_val_en_df.reset_index(drop=True))


def prep_train_sq(batch):
    enc = sq_tok(
        batch["question"],
        batch["context"],
        truncation="only_second",
        max_length=MAX_LEN,
        stride=STRIDE,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length"
    )

    sample_map= enc.pop("overflow_to_sample_mapping")
    offsets_list= enc.pop("offset_mapping")

    starts, ends = [], []

    for i, offsets in enumerate(offsets_list):
        ex = sample_map[i]
        if not batch["answerable"][ex]:
            starts.append(0)
            ends.append(0)
            continue

        s_char= batch["answer_start"][ex]
        e_char= s_char + len(batch["answer"][ex])

        seq_ids= enc.sequence_ids(i)
        ctx_s= seq_ids.index(1)
        ctx_e= len(seq_ids) - 1 - seq_ids[::-1].index(1)

        if offsets[ctx_s][0] > s_char or offsets[ctx_e][1] < e_char:
            starts.append(0)
            ends.append(0)
            continue

        ts= ctx_s
        while ts <= ctx_e and offsets[ts][0] <= s_char:
            ts += 1

        te= ctx_e
        while te >= ctx_s and offsets[te][1] >= e_char:
            te -= 1

        starts.append(ts-1)
        ends.append(te+1)

    enc["start_positions"]= starts
    enc["end_positions"]= ends
    return enc


def prep_eval_sq(batch):
    enc= sq_tok(
        batch["question"],
        batch["context"],
        truncation="only_second",
        max_length=MAX_LEN,
        stride=STRIDE,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length"
    )

    sample_map= enc.pop("overflow_to_sample_mapping")

    enc["example_id"]= [batch["id"][s] for s in sample_map]
    enc["offset_mapping"]= [[offset if enc.sequence_ids(i)[k] == 1 else None for k, offset in enumerate(offsets)]
        for i, offsets in enumerate(enc["offset_mapping"])]

    return enc


train_feats_sq= qa_train_en_ds.map(prep_train_sq,batched=True,remove_columns=qa_train_en_ds.column_names,load_from_cache_file=False)
val_feats_sq= qa_val_en_ds.map(prep_eval_sq,batched=True,remove_columns=qa_val_en_ds.column_names,load_from_cache_file=False)

print("Training examples:", len(qa_train_en_ds))
print("Training windows:", len(train_feats_sq))
print("Validation examples:", len(qa_val_en_ds))
print("Validation windows:", len(val_feats_sq))

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Map:   0%|          | 0/6335 [00:00<?, ? examples/s]

Map:   0%|          | 0/1155 [00:00<?, ? examples/s]

Training examples: 6335
Training windows: 6483
Validation examples: 1155
Validation windows: 1183


In [70]:
def run_qa(model, feats, examples):
    tr= Trainer(model=model, args=pred_args, data_collator=default_data_collator)
    s, e= tr.predict(feats.remove_columns(["example_id", "offset_mapping"])).predictions
    return postprocess(examples, feats, s, e)

In [71]:
def sq_dir(seed):
    return CACHE_DIR / f"qa_robertabase_translated_seed{seed}"
def train_sq(seed=42):
    d = sq_dir(seed)
    if d.exists():
        return AutoModelForQuestionAnswering.from_pretrained(d)
    set_seed(seed)
    model = AutoModelForQuestionAnswering.from_pretrained(SQ)
    BATCH, EPOCHS = 16, 3
    steps = int(np.ceil(len(train_feats_sq) / BATCH)) * EPOCHS
    args = TrainingArguments(output_dir=str(CACHE_DIR / "qa_robertabase_runs"),
        per_device_train_batch_size=BATCH, learning_rate=3e-5, num_train_epochs=EPOCHS,
        weight_decay=0.01, warmup_steps=int(0.1 * steps), fp16=torch.cuda.is_available(),
        seed=seed, save_strategy="no", report_to="none")
    Trainer(model=model, args=args, train_dataset=train_feats_sq,
            data_collator=default_data_collator).train()
    model.save_pretrained(d)
    return model

ga = qa_val_df["answerable"].values.astype(bool)
all_lang = qa_val_df["lang"].values
ans_lang = qa_val_df.loc[ga, "lang"].values

def per_example_ans_f1(raw):
    return np.array([_f1(raw[i]["text"], answer)
        for i, answer in zip(
            qa_val_df.loc[ga, "id"],
            qa_val_df.loc[ga, "answer"],)])

def per_example_v2_f1(raw, thr):
    scores= []
    for i, answer, is_answerable in zip(qa_val_df["id"],qa_val_df["answer"],ga,):
        if raw[i]["no_answer_score"] > thr:
            score = 0.0 if is_answerable else 1.0
        else:
            score = (_f1(raw[i]["text"], answer)if is_answerable else 0.0)
        scores.append(score)
    return np.array(scores)

SYSTEMS= ["EN roberta-base", "XLM-R"]
xlmr_res = res.loc[res["model"] == "xlmr"].copy()

rows = [xlmr_res.assign(model="XLM-R")]
ans_f1= {name: [] for name in SYSTEMS}
v2_f1= {name: [] for name in SYSTEMS}

#English roberta-base fine-tuned on translated questions + English context, 3 seeds
for seed in [42, 43, 44]:
    model = train_sq(seed)
    raw = run_qa(model,val_feats_sq,qa_val_en_ds,)
    tab, threshold, _ = evaluate(raw)
    rows.append(tab.assign(model="EN roberta-base",seed=seed,threshold=threshold,))
    ans_f1["EN roberta-base"].append(per_example_ans_f1(raw))
    v2_f1["EN roberta-base"].append(per_example_v2_f1(raw, threshold))
    del model, raw
    torch.cuda.empty_cache()

for seed in [42, 43, 44]:
    model = train_or_load("xlmr", seed)
    raw = predict_raw(model, "xlmr")
    tab, threshold, _ = evaluate(raw)
    ans_f1["XLM-R"].append(per_example_ans_f1(raw))
    v2_f1["XLM-R"].append(per_example_v2_f1(raw, threshold))
    del model, raw
    torch.cuda.empty_cache()

allres = pd.concat(rows)
cols = ["F1", "EM", "ans_macro_f1", "ans_F1", "ans_EM"]
g = allres.groupby(["model", "lang"])
mean, std = g[cols].mean(), g[cols].std()
summary = pd.DataFrame(index=mean.index)
for c in cols:
    summary[c] = [f"{a:.4f} ± {b:.4f}" for a, b in zip(mean[c], std[c])]
print(summary.to_string())

print("\nPer-run thresholds:")
print(allres[allres.lang == "All"].pivot(index="model", columns="seed", values="threshold").round(3))
print("\nPer-seed SQuAD-v2 F1 (All):")
print(allres[allres.lang == "All"].pivot(index="model", columns="seed", values="F1").round(4))
print("\nPer-seed ans_F1 (All):")
print(allres[allres.lang == "All"].pivot(index="model", columns="seed", values="ans_F1").round(4))

#Paired bootstrap over the 3 seeds per model
rng = np.random.default_rng(0)
def boot(diff, n=2000):
    idx = rng.integers(0, len(diff), (n, len(diff)))
    means = diff[idx].mean(1)
    return diff.mean(), *np.percentile(means, [2.5, 97.5])

def report(label, scores, langs):
    d = np.mean(scores["EN roberta-base"], 0) - np.mean(scores["XLM-R"], 0)
    m, lo, hi = boot(d)
    print(f"\n{label}\n  All: d = {m:+.3f}, 95% CI [{lo:+.3f}, {hi:+.3f}] (n={len(d)})")
    for lang in ["ar", "ko", "te"]:
        m, lo, hi = boot(d[langs == lang])
        print(f"  {lang}: d = {m:+.3f}, 95% CI [{lo:+.3f}, {hi:+.3f}] (n={(langs == lang).sum()})")

report("PRIMARY: SQuAD-v2 F1, all examples", v2_f1, all_lang)
report("SECONDARY: answerable-only token F1", ans_f1, ans_lang)

print("\nPer-seed d SQuAD-v2 F1:",
      [round(a.mean() - b.mean(), 3) for a, b in zip(v2_f1["EN roberta-base"], v2_f1["XLM-R"])])
print("Per-seed d ans_F1:",
      [round(a.mean() - b.mean(), 3) for a, b in zip(ans_f1["EN roberta-base"], ans_f1["XLM-R"])])

model.safetensors: reconstructing file:   0%|          |  0.00B /  499MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

{'loss': '2.176', 'grad_norm': '28.59', 'learning_rate': '1.966e-05', 'epoch': '1.232'}
{'loss': '1.025', 'grad_norm': '39.67', 'learning_rate': '5.989e-06', 'epoch': '2.463'}
{'train_runtime': '347', 'train_samples_per_second': '56.05', 'train_steps_per_second': '3.51', 'train_loss': '1.46', 'epoch': '3'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

                                   F1               EM     ans_macro_f1           ans_F1           ans_EM
model           lang                                                                                     
EN roberta-base All   0.6875 ± 0.0081  0.5937 ± 0.0048  0.7991 ± 0.0137  0.7198 ± 0.0067  0.6034 ± 0.0061
                ar    0.7271 ± 0.0075  0.6345 ± 0.0037  0.9011 ± 0.0260  0.7176 ± 0.0039  0.6051 ± 0.0069
                ko    0.7127 ± 0.0227  0.5983 ± 0.0203  0.8291 ± 0.0472  0.7257 ± 0.0179  0.5974 ± 0.0208
                te    0.6215 ± 0.0125  0.5451 ± 0.0091  0.6888 ± 0.0231  0.7159 ± 0.0067  0.6082 ± 0.0034
XLM-R           All   0.6311 ± 0.0230  0.5420 ± 0.0173  0.8072 ± 0.0446  0.6388 ± 0.0112  0.5230 ± 0.0067
                ar    0.6386 ± 0.0156  0.5510 ± 0.0100  0.8642 ± 0.0299  0.6139 ± 0.0149  0.5023 ± 0.0104
                ko    0.6708 ± 0.0058  0.5674 ± 0.0197  0.8034 ± 0.0273  0.6753 ± 0.0130  0.5539 ± 0.0181
                te    0.5863 ± 0.0559  0.5087 

# Week 44: Challenge Set

In [72]:
from google.colab import files
uploaded = files.upload()

Saving challenge.json to challenge.json


In [73]:
import pandas as pd
import json

with open('challenge.json', 'r', encoding='utf-8') as f:
    challenge_data = json.load(f)

# Convert to a pandas DataFrame
challenge_df = pd.DataFrame(challenge_data)
print(f"Loaded {len(challenge_df)} instances from challenge.json")
print("Columns:", challenge_df.columns.tolist())
print(challenge_df.head())

Loaded 20 instances from challenge.json
Columns: ['id', 'pair_id', 'created_by', 'question', 'context', 'lang', 'answerable', 'answer_start', 'answer', 'source_url']
             id     pair_id created_by                             question  \
0  andreas-01-a  andreas-01    Andreas  케카우메노스에 따르면 바실리오스 2세는 몇 명의 포로를 잡았나?   
1  andreas-01-u  andreas-01    Andreas  즐라타르스키에 따르면 바실리오스 2세는 몇 명의 포로를 잡았나?   
2  andreas-02-a  andreas-02    Andreas    متى دخل نقفور فوقاس مدينة الخندق؟   
3  andreas-02-u  andreas-02    Andreas   متى غادر نقفور فوقاس مدينة الخندق؟   
4  andreas-03-a  andreas-03    Andreas              주술회전 연재는 휴재 후 언제 재개되었나?   

                                             context lang  answerable  \
0  Skylitzes records that Basil completely routed...   ko        True   
1  Skylitzes records that Basil completely routed...   ko       False   
2  At the recommendation of the influential minis...   ar        True   
3  At the recommendation of the influential minis...   ar       Fal

In [76]:
# Week 44: frozen models on the challenge set (no training, validation thresholds reused)
# Restore the NLLB tokenizer (cell 48's loop variable overwrites the global `tok`)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tok = AutoTokenizer.from_pretrained("facebook/nllb-200-distilled-600M")
model = AutoModelForSeq2SeqLM.from_pretrained("facebook/nllb-200-distilled-600M").to(device)
model.eval()

def translate(token, lang):
    try:
        tok.src_lang = LANG_MAP[lang]
        inputs = tok(token, return_tensors="pt").to(device)
        with torch.no_grad():
            output = model.generate(
                **inputs,
                forced_bos_token_id=tok.convert_tokens_to_ids("eng_Latn"))
        return tok.decode(output[0], skip_special_tokens=True)
    except Exception as e:
        print(f"failed: {token!r} ({lang}) -> {type(e).__name__}: {e}")
        return None

ch = pd.DataFrame(json.load(open("challenge.json", encoding="utf-8")))
ch["answerable"] = ch["answerable"].astype(bool)

for q, l in zip(ch["question"], ch["lang"]):
    if translation_cache.get((q, l)) is None:
        translation_cache.pop((q, l), None)

ch["question_en"] = [ translate_cached(q, l) or ""for q, l in zip(ch["question"], ch["lang"])]
assert (ch["question_en"] != "").all(), "translation failed, check the printed errors"
del model
torch.cuda.empty_cache()

pairs = [(pid, g.index[g["answerable"]][0], g.index[~g["answerable"]][0]) for pid, g in ch.groupby("pair_id", sort=False)]

ds = Dataset.from_pandas(ch[["id", "question", "context"]])
ds_en = Dataset.from_pandas(ch[["id", "question_en", "context"]].rename(columns={"question_en": "question"}))
feats, feats_en = (d.map(prep_eval, batched=True, remove_columns=d.column_names) for d in (ds, ds_en))
feats_en_sq = ds_en.map(prep_eval_sq, batched=True, remove_columns=ds_en.column_names)

m_xlmr = train_or_load("xlmr", 42)
m_sc = train_or_load("sc", 42)
m_rb= train_sq(42)

raw_val_xlmr = predict_raw(m_xlmr, "xlmr")
_, qa_threshold, _ = evaluate(raw_val_xlmr)

raw_val_sc = predict_raw(m_sc, "sc")
_, sc_thr, _ = evaluate(raw_val_sc)

raw_val_rb = run_qa(m_rb, val_feats_sq, qa_val_en_ds)
_, thr_rb, _ = evaluate(raw_val_rb)

def qa(kind, ckpt, d, f, thr):
    out = Trainer(model=ckpt, args=pred_args, data_collator=default_data_collator).predict(
        f.remove_columns(["example_id", "offset_mapping"])).predictions
    raw = (postprocess if kind == "xlmr" else postprocess_sc)(d, f, *out)
    return {"ans": np.array([raw[i]["no_answer_score"] <= thr for i in ch["id"]]), "text": [raw[i]["text"] for i in ch["id"]], "na": np.array([raw[i]["no_answer_score"] for i in ch["id"]])}

def qc(r):
    t, o = tokenize_with_offsets(r["context"])
    return predict_span_qc_classifier({"tokens": t, "offsets": o, "context": r["context"], "question_en": r["question_en"]},
                                      vectorizer, qc_classifier)

qc_out = [qc(r) for r in ch.to_dict("records")]

ch_rule_feats = extract_features(ch)
ch_rule_pred = predict_from_features(ch_rule_feats, BEST_THRESHOLD, BEST_MODE)

P = {"Rule-based (W36)": {"ans": ch_rule_pred},
     "BiLSTM (W39)": {"ans": predict(ch)},
     "QC labeller (W37)": {"ans": np.array([s is not None for s, _ in qc_out]), "text": [t for _, t in qc_out]},
     "XLM-R QA": qa("xlmr", train_or_load("xlmr", 42), ds, feats, qa_threshold),
     "XLM-R start-cond.": qa("sc", train_or_load("sc", 42), ds, feats,sc_thr),
     "EN roberta-base (translated Q)": qa("xlmr", m_rb, ds_en, feats_en_sq, thr_rb)}

del m_xlmr, m_sc, m_rb, raw_val_xlmr, raw_val_sc, raw_val_rb
torch.cuda.empty_cache()

def outcome(p, ia, iu):
    ans, errs = np.asarray(p["ans"]).astype(bool), []
    if not ans[ia]: errs.append("false abstention")
    elif "text" in p and _norm(p["text"][ia]) != _norm(ch.loc[ia, "answer"]): errs.append("wrong span")
    if ans[iu]: errs.append("answered unanswerable")
    return " + ".join(errs) or "correct"

per_pair = pd.DataFrame({name: [outcome(p, ia, iu) for _, ia, iu in pairs] for name, p in P.items()},
                        index=[f"{pid} ({ch.loc[ia, 'lang']})" for pid, ia, _ in pairs]).T
summary = pd.DataFrame({
    "Pairs correct": per_pair.eq("correct").sum(1),
    "Same decision on both": [sum(bool(p["ans"][ia]) == bool(p["ans"][iu]) for _, ia, iu in pairs) for p in P.values()],
    **{e: per_pair.apply(lambda r: r.str.contains(e).sum(), axis=1)
       for e in ["false abstention", "wrong span", "answered unanswerable"]}})
print(f"{len(ch)} records, {len(pairs)} pairs (counts out of {len(pairs)})\n")
print(summary.to_string(), "\n")
print(per_pair.to_string())
for name in ["XLM-R QA", "XLM-R start-cond.", "EN roberta-base (translated Q)"]:
    p = P[name]
    print(f"\n{name}")
    for pid, ia, iu in pairs:
        print(f"  {pid}: pred {p['text'][ia]!r} (gold {ch.loc[ia, 'answer']!r}) | unans pred {p['text'][iu]!r} "
              f"| na ans {p['na'][ia]:+.2f} -> unans {p['na'][iu]:+.2f}")

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

20 records, 10 pairs (counts out of 10)

                                Pairs correct  Same decision on both  false abstention  wrong span  answered unanswerable
Rule-based (W36)                            1                      9                 1           0                      8
BiLSTM (W39)                                1                      9                 0           0                      9
QC labeller (W37)                           0                     10                 2           7                      8
XLM-R QA                                    0                      9                 0           5                      9
XLM-R start-cond.                           0                      9                 0           6                      9
EN roberta-base (translated Q)              1                      8                 1           3                      9 

                                                   andreas-01 (ko)                     andreas-02 (ar) 

NOTE na ans and unans are the scores that correspons to either abstaining or picking best span. If negative (like all the cases in my 5 pairs) then we pick the best span. if > 0 then we abstain. None abstained in my case, but some did get closer to abstention (especially with the translated question XLM-R). The reason for this behavior is probably that the model learned that if the context is nonsensical then we abstain. But since we keep the same context and only alter the question slighty the model is basically tricked into never abstaining, since those conditions nearly always meant answerable in training.

Other than abstention the model performs relatively well in guessing the right span for answerable questions. Apart from Arab Spring (Q5) and Q(1). Q(1) is a trick question, but Q(5) is simpler, so to explain that result i would probably say its due to the classic keyword match between question and context that makes the answer contain the name of the person that the question mentions.

When you write the report feel free to state otherwise, but for me that is the best hypothesis i can give on the results.